# Pi0.5 transcoder training — real inference trajectories, live metrics

Train a **fresh set of 18 action-expert transcoders** using the paper-informed recipe that improved the controlled pilot. Each teacher call captures all requested layers; dictionaries are optimized one at a time to limit GPU memory. This uses recorded LIBERO observations throughout episodes, with fresh Gaussian seeds and real 10-step denoising trajectories. No robot actions are executed.

**Run all cells on a high-RAM L4/A100.** Set `LAYER_IDS="8"` for a smaller single-layer run. The existing 7.39-GiB checkpoint is neither required nor overwritten. All experiment code is embedded below and executes in this notebook thread; external commands are used only for dependency installation.

| Default | Value |
|---|---:|
| Layers | 0–17 |
| Fresh activation-buffer cycles | 256 |
| Observation frames per cycle | 64 |
| Denoising calls / frame | 10 |
| Activation records / frame / layer | 500 for the current 50-token policy |
| Records per fresh buffer / layer | 32,000 |
| Optimizer batch | 1,024 records, sampled with replacement |
| Updates per cycle / layer | 32 |
| Total sampled records / layer | 8,388,608 |
| Train / validation / test split | 80 / 10 / 10% by episode |
| Validation frames / final test frames | 24 / 16, fixed before training |

This is a larger training run, **not a two-hour completion promise**. The dashboard estimates remaining time from measured cycle throughput. Session budgets pause work at safe boundaries and save resumable state; checkpoint upload and an in-flight operation may exceed the timer. Setup, model loading and dataset hashing occur before the training-session timer. Use the same run name and controls to resume. Runtime or protocol changes require a new run name.

## Recipe and evaluation labels

Reference: [DifFRACT Appendix E](https://arxiv.org/html/2606.15796v1#A5). We use trajectory harvesting, 16× ReLU dictionaries, the adapted 256-wide time network, decoder-based encoder initialization, unit decoder columns, AdamW with zero weight decay, and cosine decay. Loss: minibatch-variance-normalized reconstruction plus λ times L1 feature activity. The VLA-specific embedding, λ=1e−4, clipping, smaller buffers/batches and total training budget remain adaptations—not an exact paper reproduction.

**Fidelity is not attribution.** Reconstruction error, sparsity and normalized-action error are labelled separately. No metric here establishes semantic features, correct circuit tracing, successful steering, task success or collision prevention. Episode holdout does not imply unseen tasks or exclusion from policy pretraining.

Best measured **validation nMSE per layer** selects exported weights. Final test frames never affect optimization or selection. The final report measures each layer separately; it does not certify simultaneous replacement of all layers.


In [ ]:
# @title Controls — Freeze These For A Resumable Run
from pathlib import Path
DRIVE_ROOT = '/content/drive/MyDrive/groot-run-shared-programmer908' # @param {type:"string"}
RUN_NAME = 'pi05-trajectory-transcoders-001' # @param {type:"string"}
LAYER_IDS = ','.join(str(i) for i in range(18)) # @param {type:"string"}
CYCLES = 256 # @param {type:"integer"}
FRAMES_PER_CYCLE = 64 # @param {type:"integer"}
TRAIN_BATCH_SIZE = 1024 # @param {type:"integer"}
LAMBDA_L1 = 0.0001 # @param {type:"number"}
VALIDATE_EVERY_CYCLES = 8 # @param {type:"integer"}
SESSION_MINUTES = 120 # @param {type:"integer"}
CHECKPOINT_EVERY_MINUTES = 30 # @param {type:"integer"}
DATASET_REPO = 'HuggingFaceVLA/libero' # @param {type:"string"}
DATASET_ROOT = '' # @param {type:"string"}
ALLOW_DATASET_DOWNLOAD = True # @param {type:"boolean"}
HF_OFFLINE = True # @param {type:"boolean"}
ALLOW_AUTH_REFRESH = True # @param {type:"boolean"}
POLICY_REPO = 'lerobot/pi05_libero_finetuned'
POLICY_REVISION = 'main'
if not RUN_NAME or Path(RUN_NAME).name!=RUN_NAME: raise ValueError('Use a single run-folder name')


In [ ]:
# @title Mount Drive — New Output Folder, Existing Checkpoints Preserved
from google.colab import drive
if not Path('/content/drive/MyDrive').exists(): drive.mount('/content/drive')
DRIVE_ROOT = Path(DRIVE_ROOT)
if not DRIVE_ROOT.is_dir(): raise FileNotFoundError(DRIVE_ROOT)
RUN_DIR = DRIVE_ROOT/'outputs/trajectory-training'/RUN_NAME
RUN_DIR.mkdir(parents=True,exist_ok=True)
print('Persistent training output:',RUN_DIR)
print('Resume pointer:',RUN_DIR/'latest.json')


In [ ]:
# @title Reuse Or Install The Pinned Runtime
if True:
    if not Path("/dev/nvidia0").exists():
        raise RuntimeError("Select a high-RAM L4/A100 GPU runtime for training.")
    # @title Install Pinned LeRobot Runtime (Repairs Existing Environment)

    import os
    import sys
    import subprocess
    from pathlib import Path

    # Rerunning this cell cannot unload native modules. Avoid a reinstall loop.
    install_receipt = Path('/content/pi05-audit-install-receipt.json')
    if install_receipt.exists():
        import json
        if json.loads(install_receipt.read_text()).get('pid') == os.getpid():
            raise RuntimeError('Packages are already installed. Use Runtime > Restart session once, then Run all; do not delete the runtime.')

    # Set cache and rendering variables before importing Transformers or LeRobot;
    # these libraries cache environment settings at import time in the kernel.
    os.environ.update({
        "HF_HOME": "/content/hf_home",
        "HF_HUB_CACHE": "/content/hf_home/hub",
        "MUJOCO_GL": "egl",
        "PYOPENGL_PLATFORM": "egl",
        "MUJOCO_EGL_DEVICE_ID": "0",
        "MPLBACKEND": "Agg",
        "LIBERO_CONFIG_PATH": "/content/pi05-audit-support/.libero",
        "LIBERO_DATASET_DIR": "/content/pi05-audit-support/data/libero/datasets",
    })
    for key in ("HF_HUB_OFFLINE", "TRANSFORMERS_OFFLINE", "HF_DATASETS_OFFLINE"):
        if HF_OFFLINE:
            os.environ[key] = "1"
        else:
            os.environ.pop(key, None)
    loaded_hub = sys.modules.get("huggingface_hub.constants")
    if loaded_hub is not None and loaded_hub.HF_HUB_CACHE != os.environ["HF_HUB_CACHE"]:
        raise RuntimeError("Restart the Colab session, then rerun from the top to use the configured model cache.")

    PYTHON = Path(sys.executable)  # Install into the kernel used for direct collection.
    UV_BIN_DIR = Path("/content/uv-bin")
    UV = str(UV_BIN_DIR / "uv")


    def run(cmd, *, env=None):
        # External setup tools (uv/apt/git), never a Python collection worker.
        # Stream stderr too: inherited child output may be absent from Colab's cell.
        from collections import deque
        import shlex
        cmd = list(map(str, cmd))
        print("$", shlex.join(cmd), flush=True)
        tail = deque(maxlen=100)
        with subprocess.Popen(cmd, env=env, stdout=subprocess.PIPE,
                              stderr=subprocess.STDOUT, text=True, bufsize=1) as process:
            try:
                for line in process.stdout:
                    print(line, end="", flush=True)
                    tail.append(line)
                returncode = process.wait()
            except BaseException:
                process.terminate()
                try:
                    process.wait(timeout=5)
                except subprocess.TimeoutExpired:
                    process.kill()
                    process.wait()
                raise
        if returncode:
            detail = "".join(tail).strip() or "The command produced no output."
            raise RuntimeError(f"Setup command failed (exit {returncode}): {shlex.join(cmd)}\n\n{detail}")
        return subprocess.CompletedProcess(cmd, returncode)


    def install_uv() -> str:
        # Colab may already have /usr/local/bin/uv, but its version/behavior can differ.
        # Use a notebook-owned binary so every clean runtime follows the same path.
        UV_BIN_DIR.mkdir(parents=True, exist_ok=True)
        installer = Path("/tmp/install-uv.sh")
        run(["curl", "-LsSf", "https://astral.sh/uv/install.sh", "-o", installer])
        env = os.environ.copy()
        env["UV_INSTALL_DIR"] = str(UV_BIN_DIR)
        run(["sh", installer], env=env)
        os.environ["PATH"] = f"{UV_BIN_DIR}:" + os.environ["PATH"]
        run([UV, "--version"])
        return UV


    # Check in this kernel; do not mix a Python 3.12 venv into a Python 3.13 kernel.
    # Package installation cannot replace native modules already loaded by this process.
    loaded_runtime = any(name in sys.modules for name in ("torch", "numpy", "transformers", "lerobot"))
    try:
        import importlib.util
        from importlib.metadata import version
        assert version('lerobot') == '0.4.4'
        assert version('torch').split('+')[0] == '2.10.0'
        assert version('torchvision').split('+')[0] == '0.25.0'
        from transformers.models.siglip import check
        assert check.check_whether_transformers_replace_is_installed_correctly()
        import torch
        assert torch.cuda.is_available()
        assert all(importlib.util.find_spec(name) for name in ('libero', 'langfuse', 'sklearn', 'joblib', 'PIL', 'matplotlib'))
        ready = True
    except (ImportError, AssertionError, ValueError, RuntimeError) as exc:
        print("Runtime needs installation:", exc)
        ready = False
    if ready:
        print("Reusing existing CUDA/LeRobot runtime:", PYTHON)
        if not Path(UV).exists():
            UV = install_uv()
    else:
        apt_packages = [
            "build-essential", "ca-certificates", "cmake", "curl", "ffmpeg", "git", "pv", "rsync",
            "libegl1", "libgl1", "libglib2.0-0", "libglvnd0", "libglx0", "libopengl0",
            "libosmesa6-dev", "libsm6", "libxext6", "libxrender1", "pkg-config",
        ]
        apt_env = os.environ.copy()
        apt_env["DEBIAN_FRONTEND"] = "noninteractive"
        run(["apt-get", "update", "-qq"], env=apt_env)
        run(["apt-get", "install", "-y", "-qq", *apt_packages], env=apt_env)
        UV = install_uv()

        # egl-probe's source build predates CMake 4. Install its build tools first.
        run([UV, "pip", "install", "--python", str(PYTHON), "--system", "--break-system-packages",
             "cmake==3.31.6", "setuptools>=71,<81", "wheel"])
        import sysconfig
        os.environ["PATH"] = sysconfig.get_path("scripts") + os.pathsep + os.environ["PATH"]
        run(["cmake", "--version"])
        run([UV, "pip", "install", "--python", str(PYTHON), "--system", "--break-system-packages",
             "--no-build-isolation-package", "egl-probe", "egl-probe==1.0.2"])

        # Match scripts/pi05_transformers_overrides.txt; the repo is cloned in the next cell.
        transformers_override = Path("/content/pi05-transformers-overrides.txt")
        transformers_override.write_text('transformers @ git+https://github.com/huggingface/transformers.git@dcddb970176382c0fcf4521b0c0e6fc15894dfe0' + "\n")

        run([
            UV, "pip", "install", "--python", str(PYTHON), "--system", "--break-system-packages",
            "--torch-backend", "cu128",
            "--overrides", str(transformers_override),
            "--no-build-isolation-package", "egl-probe", "cmake==3.31.6",
            "lerobot[libero]==0.4.4", "hf-transfer", "opencv-python", "numpy", "langfuse",
            "scikit-learn", "joblib", "pillow", "matplotlib", "torch==2.10.0", "torchvision==0.25.0",
        ])

        import json
        install_receipt.write_text(json.dumps({'pid': os.getpid()}))
        print("Installed into notebook Python:", PYTHON)
        # The readiness probe itself may also have imported modules before failing.
        if loaded_runtime or any(name in sys.modules for name in ("torch", "numpy", "transformers", "lerobot")):
            raise RuntimeError("Packages installed. Restart the Colab session once, then rerun the notebook. "
                               "This clears old Python/native modules; Drive caches are preserved.")

else:
    import torch, numpy
    print("Using current torch for metadata inspection only; no package replacement.")


In [ ]:
# @title Ensure Charts And Video Decoder Dependencies
import importlib.util, subprocess, sys
missing=[p for p in ('matplotlib','av') if importlib.util.find_spec(p) is None]
if missing:
    subprocess.run([sys.executable,'-m','pip','install',*missing],check=True)
import matplotlib
matplotlib.use('Agg')
print('Chart backend:',matplotlib.get_backend(),'; video decoder: PyAV')


In [ ]:
# @title Prepare Model Cache And LIBERO Assets (Skipped For Metadata Only)
import os, shutil, site, tarfile
from pathlib import Path
snapshots = {}
if True:
    cache=Path('/content/hf_home'); cache.mkdir(parents=True,exist_ok=True)
    archive=DRIVE_ROOT/'archives/hf_home.tar'
    policy_cache=cache/'hub'/('models--'+POLICY_REPO.replace('/','--'))/'snapshots'
    policy_cached=policy_cache.is_dir() and any((p/'config.json').is_file() and
        ((p/'model.safetensors').is_file() or (p/'model.safetensors.index.json').is_file()) for p in policy_cache.iterdir())
    if not policy_cached:
        if archive.is_file():
            print('Restoring model cache archive',flush=True)
            with tarfile.open(archive) as handle: handle.extractall(cache,filter='data')
        elif (DRIVE_ROOT/'hf_home').is_dir():
            print('Copying existing Drive model cache',flush=True)
            shutil.copytree(DRIVE_ROOT/'hf_home',cache,dirs_exist_ok=True)
    from huggingface_hub import snapshot_download, constants as hf_constants
    for repo,kind in [(POLICY_REPO,'model'),('google/paligemma-3b-pt-224','model'),('lerobot/libero-assets','dataset')]:
        revision=POLICY_REVISION if repo==POLICY_REPO else 'main'
        try:
            path=snapshot_download(repo,repo_type=kind,revision=revision,local_files_only=True)
        except Exception:
            if not ALLOW_AUTH_REFRESH: raise
            token=os.environ.get('HF_TOKEN')
            if not token:
                from google.colab import userdata
                try: token=userdata.get('HF_TOKEN')
                except Exception: token=None
            old=hf_constants.HF_HUB_OFFLINE; hf_constants.HF_HUB_OFFLINE=False
            try: path=snapshot_download(repo,repo_type=kind,revision=revision,token=token,local_files_only=False)
            finally: hf_constants.HF_HUB_OFFLINE=old
        snapshots[repo]=str(Path(path).resolve())
        print(repo,snapshots[repo],flush=True)
    roots=[Path(p)/'libero/libero' for p in site.getsitepackages()]
    libero_root=next((p for p in roots if p.is_dir()),None)
    if libero_root is None: raise RuntimeError('hf-libero is missing; rerun runtime setup')
    assets=libero_root/'assets'
    if not (assets/'scenes/libero_tabletop_base_style.xml').exists():
        shutil.copytree(snapshots['lerobot/libero-assets'],assets,dirs_exist_ok=True)
    if not (assets/'scenes/libero_tabletop_base_style.xml').exists(): raise RuntimeError('LIBERO assets are incomplete')
    Path('/content/pi05-audit-support').mkdir(exist_ok=True)
    import torch
    if not torch.cuda.is_available(): raise RuntimeError('Select a GPU runtime for training')
    ram_gib=int(next(line.split()[1] for line in Path('/proc/meminfo').read_text().splitlines() if line.startswith('MemTotal:')))/2**20
    gpu_gib=torch.cuda.get_device_properties(0).total_memory/2**30
    print(f'GPU memory: {gpu_gib:.1f} GiB; system memory: {ram_gib:.1f} GiB')
    if gpu_gib<21 or ram_gib<23: raise RuntimeError('Use a high-RAM L4/A100: at least 22 GiB GPU and 24 GiB system RAM recommended')
else:
    print('Metadata-only: no simulator, policy, assets or GPU required.')


In [ ]:
# @title Open The Full Observation Dataset (No DataLoader Worker Processes)
import os,shutil
from huggingface_hub import constants as hf_constants
from lerobot.datasets.lerobot_dataset import LeRobotDataset
from lerobot.utils.constants import HF_LEROBOT_HOME
if DATASET_ROOT.strip():
    dataset_root=Path(DATASET_ROOT).expanduser()
else:
    dataset_root=Path('/content/lerobot-data')/DATASET_REPO
    candidates=[Path(HF_LEROBOT_HOME)/DATASET_REPO,Path('/content/hf_home/lerobot')/DATASET_REPO,
                DRIVE_ROOT/'hf_home/lerobot'/DATASET_REPO,DRIVE_ROOT/'libero_cache'/DATASET_REPO]
    if not (dataset_root/'meta/info.json').exists():
        cached=next((p for p in candidates if (p/'meta/info.json').is_file()),None)
        if cached is not None and cached.resolve()!=dataset_root.resolve():
            print('Staging cached dataset on local disk:',cached,flush=True)
            dataset_root.mkdir(parents=True,exist_ok=True)
            for name in ('meta','data','videos','images'):
                if (cached/name).is_dir(): shutil.copytree(cached/name,dataset_root/name,dirs_exist_ok=True)
print('Dataset:',DATASET_REPO,'root:',dataset_root,flush=True)
old_offline=hf_constants.HF_HUB_OFFLINE
old_env={k:os.environ.get(k) for k in ('HF_HUB_OFFLINE','HF_DATASETS_OFFLINE')}
try:
    if ALLOW_DATASET_DOWNLOAD:
        hf_constants.HF_HUB_OFFLINE=False
        for key in old_env: os.environ.pop(key,None)
    dataset=LeRobotDataset(DATASET_REPO,root=dataset_root,video_backend='pyav',delta_timestamps=None)
finally:
    hf_constants.HF_HUB_OFFLINE=old_offline
    for key,value in old_env.items():
        if value is None: os.environ.pop(key,None)
        else: os.environ[key]=value
print('Dataset frames:',len(dataset),'; episodes:',dataset.meta.total_episodes)
print('Camera keys:',dataset.meta.camera_keys)
print('No simulator rollout, occlusion masks, or bowl inference is used for training.')


In [ ]:
# @title Definitions 1 — Exact Dictionary And Policy Guards
"""DifFRACT-style timestep-conditioned transcoders.

The transcoder approximates one transformer MLP:

    TC_l(x, t) ~= MLP_l(x)

where x is one token-position hidden vector and t is the flow/diffusion
timestep. The module also supports batched token tensors shaped
``[batch, tokens, d_model]``.
"""


from dataclasses import dataclass

import torch
import torch.nn.functional as F
from torch import Tensor, nn


@dataclass(frozen=True)
class TimeConditionedTranscoderConfig:
    """Configuration for one timestep-conditioned transcoder."""

    d_model: int
    d_features: int | None = None
    expansion_factor: int = 16
    time_embedding_dim: int | None = None
    time_hidden_dim: int | None = None
    min_period: float = 4e-3
    max_period: float = 4.0
    eps: float = 1e-8

    @property
    def latent_dim(self) -> int:
        return self.d_features if self.d_features is not None else self.d_model * self.expansion_factor

    @property
    def time_dim(self) -> int:
        return self.time_embedding_dim if self.time_embedding_dim is not None else self.d_model

    @property
    def time_hidden(self) -> int:
        return self.time_hidden_dim if self.time_hidden_dim is not None else self.d_model


@dataclass(frozen=True)
class TranscoderLoss:
    """Scalar loss terms for logging and optimization."""

    total: Tensor
    normalized_mse: Tensor
    l1: Tensor


def sinusoidal_time_embedding(
    timesteps: Tensor,
    dim: int,
    *,
    min_period: float = 4e-3,
    max_period: float = 4.0,
) -> Tensor:
    """Create sinusoidal timestep embeddings for scalar flow/diffusion times.

    Args:
        timesteps: Tensor shaped ``[batch]`` or ``[batch, 1]``.
        dim: Output embedding width.
        min_period: Shortest sinusoid period.
        max_period: Longest sinusoid period.
    """
    if dim <= 0:
        raise ValueError(f"dim must be positive, got {dim}")

    t = timesteps.float().reshape(-1, 1)
    half_dim = dim // 2
    if half_dim == 0:
        return t.new_zeros((t.shape[0], dim))

    periods = torch.logspace(
        start=torch.log10(t.new_tensor(min_period)),
        end=torch.log10(t.new_tensor(max_period)),
        steps=half_dim,
        device=t.device,
        dtype=t.dtype,
    )
    phases = 2.0 * torch.pi * t / periods.unsqueeze(0)
    emb = torch.cat([torch.sin(phases), torch.cos(phases)], dim=-1)
    if emb.shape[-1] < dim:
        emb = F.pad(emb, (0, dim - emb.shape[-1]))
    return emb


def normalized_mse_loss(prediction: Tensor, target: Tensor, variance_denominator: Tensor, eps: float) -> Tensor:
    """Return variance-normalized squared reconstruction error.

    ``variance_denominator`` is the per-transcoder denominator:
    ``sum_j Var[MLP_l(x)_j]`` estimated over that MLP's activation records.
    """
    squared_error = (prediction.float() - target.float()).pow(2).sum(dim=-1).mean()
    denom = variance_denominator.float().clamp_min(eps)
    return squared_error / denom


class TimeConditionedTranscoder(nn.Module):
    """Sparse timestep-conditioned transcoder for one MLP."""

    def __init__(self, config: TimeConditionedTranscoderConfig):
        super().__init__()
        self.config = config
        d_model = config.d_model
        d_features = config.latent_dim

        self.time_mlp = nn.Sequential(
            nn.Linear(config.time_dim, config.time_hidden),
            nn.SiLU(),
            nn.Linear(config.time_hidden, 2 * d_model),
        )
        self.encoder = nn.Linear(d_model, d_features)
        self.decoder = nn.Linear(d_features, d_model)

        self.reset_parameters()

    def reset_parameters(self) -> None:
        nn.init.kaiming_uniform_(self.encoder.weight, a=5**0.5)
        nn.init.zeros_(self.encoder.bias)
        nn.init.kaiming_uniform_(self.decoder.weight, a=5**0.5)
        nn.init.zeros_(self.decoder.bias)

        last_time = self.time_mlp[-1]
        if not isinstance(last_time, nn.Linear):
            raise TypeError("Expected final time_mlp module to be nn.Linear")
        nn.init.zeros_(last_time.weight)
        nn.init.zeros_(last_time.bias)

    def _time_scale_shift(self, timesteps: Tensor, *, dtype: torch.dtype, device: torch.device) -> tuple[Tensor, Tensor]:
        time_emb = sinusoidal_time_embedding(
            timesteps.to(device=device),
            self.config.time_dim,
            min_period=self.config.min_period,
            max_period=self.config.max_period,
        ).to(dtype=dtype)
        scale_shift = self.time_mlp(time_emb)
        scale, shift = scale_shift.chunk(2, dim=-1)
        return scale, shift

    def encode(self, x: Tensor, timesteps: Tensor) -> tuple[Tensor, Tensor]:
        """Return ``(feature_preactivation, sparse_latent)``.

        Args:
            x: MLP input, shaped ``[records, d_model]`` or
                ``[batch, tokens, d_model]``.
            timesteps: Scalar timesteps shaped ``[records]`` for 2D x or
                ``[batch]`` for 3D x.
        """
        if x.shape[-1] != self.config.d_model:
            raise ValueError(f"Expected x last dim {self.config.d_model}, got {x.shape[-1]}")

        param_dtype = self.encoder.weight.dtype
        x_for_transcoder = x.to(dtype=param_dtype)
        scale, shift = self._time_scale_shift(timesteps, dtype=param_dtype, device=x.device)
        if x.ndim == 3:
            if scale.shape[0] != x.shape[0]:
                raise ValueError(f"Expected {x.shape[0]} timesteps for 3D x, got {scale.shape[0]}")
            scale = scale[:, None, :]
            shift = shift[:, None, :]
        elif x.ndim == 2:
            if scale.shape[0] != x.shape[0]:
                raise ValueError(f"Expected {x.shape[0]} timesteps for 2D x, got {scale.shape[0]}")
        else:
            raise ValueError(f"Expected x with rank 2 or 3, got shape {tuple(x.shape)}")

        x_mod = x_for_transcoder * (1.0 + scale) + shift
        preactivation = self.encoder(x_mod)
        z = F.relu(preactivation)
        return preactivation, z

    def forward(
        self,
        x: Tensor,
        timesteps: Tensor,
        *,
        return_preactivation: bool = False,
    ) -> tuple[Tensor, Tensor] | tuple[Tensor, Tensor, Tensor]:
        """Return ``(predicted_mlp_output, sparse_latent)``.

        Set ``return_preactivation=True`` when circuit tracing needs the
        encoder pre-ReLU feature values as attribution targets.
        """
        preactivation, z = self.encode(x, timesteps)
        y_hat = self.decoder(z)
        if return_preactivation:
            return y_hat, z, preactivation
        return y_hat, z

    def loss(
        self,
        x: Tensor,
        timesteps: Tensor,
        target: Tensor,
        variance_denominator: Tensor,
        *,
        lambda_l1: float,
    ) -> TranscoderLoss:
        """Compute DifFRACT-style normalized reconstruction + sparsity loss."""
        prediction, latent = self(x, timesteps)
        normalized_mse = normalized_mse_loss(
            prediction,
            target,
            variance_denominator,
            eps=self.config.eps,
        )
        l1 = latent.float().abs().sum(dim=-1).mean()
        total = normalized_mse + lambda_l1 * l1
        return TranscoderLoss(total=total, normalized_mse=normalized_mse, l1=l1)

    @torch.no_grad()
    def renormalize_decoder_columns(self, eps: float | None = None) -> None:
        """Normalize decoder feature vectors after an optimizer step.

        ``nn.Linear`` stores decoder weights as ``[d_model, d_features]``.
        Each column is the output vector for one sparse feature.
        """
        epsilon = self.config.eps if eps is None else eps
        weight = self.decoder.weight
        column_norms = weight.norm(dim=0, keepdim=True).clamp_min(epsilon)
        weight.div_(column_norms)

"""Refuse to run on a Pi0.5 whose pretrained weights did not actually load.

LeRobot's ``PI05Policy.from_pretrained`` wraps the whole state-dict load in a
``try``/``except`` that prints ``Warning: Could not load state dict`` and
returns the freshly initialised model anyway; and with ``strict=False`` a
partially matching checkpoint loads silently with missing keys. Either way the
policy runs on random weights and every downstream number is meaningless, while
nothing in the process exits non-zero.

The fix is to record what ``load_state_dict`` reported and check it after the
policy is built. A policy with no record was never loaded; a policy with
missing keys is not the checkpoint.
"""


import copy
from typing import Any

LOAD_REPORT_ATTR = "_pi05_mi_load_report"


def restore_tied_parameter_aliases(model, state_dict):
    """Restore omitted safetensors keys only for parameters the model actually shares.

    PaliGemma ties its token embedding to lm_head. Safetensors may save only
    one name; nn.Module.load_state_dict(strict=True) still requires both.
    Never invent a tensor or infer a tie from matching names or shapes.
    """
    groups = {}
    for name, parameter in model.named_parameters(remove_duplicate=False):
        groups.setdefault(id(parameter), []).append(name)
    restored = {}
    result = state_dict
    for names in groups.values():
        present = [name for name in names if name in state_dict]
        if not present:
            continue
        for name in names:
            if name not in state_dict:
                if result is state_dict:
                    result = copy.copy(state_dict)
                result[name] = state_dict[present[0]]
                restored[name] = present[0]
    return result, restored


def install_load_recorder(cls: type | None = None) -> None:
    """Make ``cls.load_state_dict`` leave a record of missing and unexpected keys.

    Defaults to LeRobot's ``PI05Policy``. Idempotent.
    """
    if cls is None:
        from lerobot.policies.pi05 import modeling_pi05

        cls = modeling_pi05.PI05Policy
    original = cls.load_state_dict
    if getattr(original, "_pi05_mi_recorder", False):
        return

    def load_state_dict_recording(self, state_dict, strict: bool = True, assign: bool = False):
        state_dict, restored = restore_tied_parameter_aliases(self, state_dict)
        # Clear any previous successful report before a later load can fail.
        setattr(self, LOAD_REPORT_ATTR, None)
        try:
            result = original(self, state_dict, strict=strict, assign=assign)
        except Exception as exc:
            setattr(self, LOAD_REPORT_ATTR, {'error': f'{type(exc).__name__}: {exc}'})
            raise
        setattr(
            self,
            LOAD_REPORT_ATTR,
            {
                "missing": list(getattr(result, "missing_keys", [])),
                "unexpected": list(getattr(result, "unexpected_keys", [])),
                "provided": len(state_dict),
                "strict": bool(strict),
                "restored_tied_aliases": restored,
            },
        )
        return result

    load_state_dict_recording._pi05_mi_recorder = True  # type: ignore[attr-defined]
    cls.load_state_dict = load_state_dict_recording


def load_report(policy: Any) -> dict[str, Any] | None:
    return getattr(policy, LOAD_REPORT_ATTR, None)


def assert_weights_loaded(policy: Any, *, source: str = "", max_listed: int = 5) -> dict[str, Any]:
    """Raise unless the policy's pretrained weights loaded completely.

    Unexpected keys (present in the checkpoint, absent from the model) are
    tolerated and returned in the report; missing keys are not, because a
    missing parameter is one running on its random initialisation.
    """
    report = load_report(policy)
    where = f" from {source}" if source else ""
    if report is None:
        raise RuntimeError(
            f"Pi0.5 weights were never loaded{where}. LeRobot swallowed the failure and returned a "
            "randomly initialised model; look for 'Could not load state dict' in the log above. "
            "Refusing to run: every measurement would be about random weights."
        )
    if report.get('error'):
        raise RuntimeError(
            f"Pi0.5 checkpoint loading failed{where}:\n{report['error']}\n"
            "Refusing to run with incomplete weights."
        )
    missing = report["missing"]
    if missing:
        shown = "\n  ".join(missing[:max_listed])
        more = f"\n  ... and {len(missing) - max_listed} more" if len(missing) > max_listed else ""
        raise RuntimeError(
            f"{len(missing)} Pi0.5 parameters did not load{where} and are running on random "
            f"initialisation:\n  {shown}{more}\n"
            "A vision-tower key layout mismatch is the usual cause. Refusing to run."
        )
    return report

"""Runtime wrappers for Pi0.5 action-expert MLP transcoders."""


import re
import types
from collections import defaultdict
from contextlib import contextmanager
from dataclasses import dataclass
from typing import Callable, Iterator, Literal

import torch
from torch import Tensor, nn



TranscoderMode = Literal["train", "probe", "replace"]

ACTION_EXPERT_MLP_RE = re.compile(r"paligemma_with_expert\.gemma_expert\.model\.layers\.(\d+)\.mlp$")


@dataclass(frozen=True)
class MLPActivationRecord:
    """One captured MLP input-output tensor for one layer call."""

    name: str
    layer_index: int
    x: Tensor
    y: Tensor
    timestep: Tensor


@dataclass(frozen=True)
class MLPTranscoderLatentRecord:
    """Compact latent summary for one transcoder call."""

    name: str
    layer_index: int
    mode: str
    timestep: Tensor
    latent_shape: tuple[int, ...]
    latent_l0: Tensor
    latent_l1: Tensor
    latent_max: Tensor
    top_indices: Tensor
    top_values: Tensor
    full_latent: Tensor | None = None


@dataclass(frozen=True)
class MLPTranscoderTraceRecord:
    """Gradient-preserving transcoder tensors from one wrapped MLP call."""

    name: str
    layer_index: int
    mode: str
    timestep: Tensor
    preactivation: Tensor
    latent: Tensor


@dataclass(frozen=True)
class ActionExpertMLPTarget:
    """One Pi0.5 action-expert MLP target."""

    name: str
    layer_index: int
    module: nn.Module
    d_model: int


class Pi05TranscoderContext:
    """Shared runtime state for Pi0.5 transcoder wrappers."""

    def __init__(
        self,
        mode: TranscoderMode = "train",
        *,
        detach_records: bool = True,
        capture_records: bool = True,
        capture_latents: bool = False,
        capture_traces: bool = False,
        latent_top_k: int = 64,
        save_full_latents: bool = False,
        latent_callback: Callable[[str, int, Tensor, Tensor], None] | None = None,
        latent_intervention: Callable[[str, int, Tensor, Tensor], Tensor | None] | None = None,
        trace_callback: Callable[[str, int, Tensor, Tensor, Tensor], None] | None = None,
        diffusion_callback: Callable[[str, Tensor, Tensor | None], None] | None = None,
        store_latent_summaries: bool = True,
        capture_diffusion: bool = False,
    ):
        self.mode = mode
        self.detach_records = detach_records
        self.capture_records = capture_records
        self.capture_latents = capture_latents
        self.capture_diffusion = capture_diffusion
        self.capture_traces = capture_traces
        self.latent_top_k = latent_top_k
        self.save_full_latents = save_full_latents
        self.latent_callback = latent_callback
        self.trace_callback = trace_callback
        # Causal intervention hook. In replace mode the wrapper offers each
        # transcoder latent to this callable; returning a tensor substitutes it
        # and the decoder is re-run, so the patched features -- and nothing
        # else -- drive the rest of the forward pass.
        self.latent_intervention = latent_intervention
        self.diffusion_callback = diffusion_callback
        self.store_latent_summaries = store_latent_summaries
        self.current_timestep: Tensor | None = None
        self.records: dict[str, list[MLPActivationRecord]] = defaultdict(list)
        self.latents: dict[str, list[MLPTranscoderLatentRecord]] = defaultdict(list)
        self.trace_records: dict[str, list[MLPTranscoderTraceRecord]] = defaultdict(list)

    @contextmanager
    def use_timestep(self, timestep: Tensor) -> Iterator[None]:
        previous = self.current_timestep
        self.current_timestep = timestep
        try:
            yield
        finally:
            self.current_timestep = previous

    def clear_records(self) -> None:
        self.records.clear()
        self.latents.clear()
        self.trace_records.clear()

    def timestep_for(self, x: Tensor) -> Tensor:
        if self.current_timestep is None:
            raise RuntimeError(
                "No Pi0.5 timestep is active. Call the wrapped model through PI05Pytorch.forward "
                "or PI05Pytorch.denoise_step so the transcoder context can see t."
            )
        timestep = self.current_timestep.to(device=x.device)
        if timestep.ndim == 0:
            timestep = timestep.reshape(1)
        if x.ndim == 3 and timestep.shape[0] != x.shape[0]:
            if timestep.numel() == 1:
                timestep = timestep.expand(x.shape[0])
            else:
                raise ValueError(f"Expected timestep batch {x.shape[0]}, got shape {tuple(timestep.shape)}")
        if x.ndim == 2 and timestep.shape[0] != x.shape[0]:
            if timestep.numel() == 1:
                timestep = timestep.expand(x.shape[0])
            else:
                raise ValueError(f"Expected one timestep per record for x shape {tuple(x.shape)}, got {tuple(timestep.shape)}")
        return timestep

    def record(self, name: str, layer_index: int, x: Tensor, y: Tensor, timestep: Tensor) -> None:
        if not self.capture_records:
            return
        if self.detach_records:
            x = x.detach()
            y = y.detach()
            timestep = timestep.detach()
        self.records[name].append(
            MLPActivationRecord(
                name=name,
                layer_index=layer_index,
                x=x,
                y=y,
                timestep=timestep,
            )
        )

    def record_trace(
        self,
        name: str,
        layer_index: int,
        preactivation: Tensor,
        latent: Tensor,
        timestep: Tensor,
    ) -> None:
        if not self.capture_traces:
            return
        if self.trace_callback is not None:
            self.trace_callback(name, layer_index, preactivation, latent, timestep)
            return
        self.trace_records[name].append(
            MLPTranscoderTraceRecord(
                name=name,
                layer_index=layer_index,
                mode=self.mode,
                timestep=timestep,
                preactivation=preactivation,
                latent=latent,
            )
        )

    def record_latent(self, name: str, layer_index: int, latent: Tensor, timestep: Tensor) -> None:
        if not self.capture_latents:
            return
        if self.latent_callback is not None:
            self.latent_callback(name, layer_index, latent, timestep)
        if not self.store_latent_summaries:
            return
        with torch.no_grad():
            z = latent.detach().float()
            timestep = timestep.detach().float().cpu()
            latent_l0 = (z > 0).sum(dim=-1).detach().cpu()
            latent_l1 = z.abs().sum(dim=-1).detach().cpu()
            latent_max = z.max(dim=-1).values.detach().cpu()

            flat = z.reshape(-1)
            top_k = min(max(0, int(self.latent_top_k)), flat.numel())
            if top_k > 0:
                top_values, top_flat_indices = torch.topk(flat, k=top_k)
                top_indices = torch.stack(torch.unravel_index(top_flat_indices.cpu(), z.shape), dim=-1)
            else:
                top_values = torch.empty(0, dtype=torch.float32)
                top_indices = torch.empty((0, z.ndim), dtype=torch.long)

            full_latent = z.cpu() if self.save_full_latents else None
            self.latents[name].append(
                MLPTranscoderLatentRecord(
                    name=name,
                    layer_index=layer_index,
                    mode=self.mode,
                    timestep=timestep,
                    latent_shape=tuple(z.shape),
                    latent_l0=latent_l0,
                    latent_l1=latent_l1,
                    latent_max=latent_max,
                    top_indices=top_indices.cpu(),
                    top_values=top_values.cpu(),
                    full_latent=full_latent,
                )
            )

    def record_diffusion(self, kind: str, value: Tensor, timestep: Tensor | None = None) -> None:
        if not self.capture_diffusion or self.diffusion_callback is None:
            return
        self.diffusion_callback(kind, value, timestep)


class WrappedActionExpertMLP(nn.Module):
    """Wrapper that preserves, observes, or replaces one Pi0.5 action-expert MLP."""

    def __init__(
        self,
        *,
        name: str,
        layer_index: int,
        original_mlp: nn.Module,
        context: Pi05TranscoderContext,
        transcoder: TimeConditionedTranscoder | None = None,
    ):
        super().__init__()
        self.name = name
        self.layer_index = layer_index
        self.original_mlp = original_mlp
        self.transcoder = transcoder
        self.context = context

    def forward(self, x: Tensor) -> Tensor:
        timestep = self.context.timestep_for(x)

        if self.context.mode == "replace":
            if self.transcoder is None:
                raise RuntimeError(f"Cannot run {self.name} in replace mode without a transcoder")
            y_hat, latent, preactivation = self.transcoder(x, timestep, return_preactivation=True)
            intervention = self.context.latent_intervention
            if intervention is not None:
                if self.context.capture_traces:
                    raise RuntimeError(
                        f"{self.name}: cannot capture traces and intervene in the same pass. "
                        "Attribution needs the unmodified computational graph, so record the "
                        "trace first and intervene in a separate forward."
                    )
                patched = intervention(self.name, self.layer_index, latent, timestep)
                if patched is not None:
                    if patched.shape != latent.shape:
                        raise ValueError(
                            f"Latent intervention on {self.name} returned shape {tuple(patched.shape)}, "
                            f"expected {tuple(latent.shape)}"
                        )
                    latent = patched
                    y_hat = self.transcoder.decoder(latent.to(dtype=self.transcoder.decoder.weight.dtype))
            else:
                self.context.record_trace(self.name, self.layer_index, preactivation, latent, timestep)
            self.context.record_latent(self.name, self.layer_index, latent, timestep)
            return y_hat.to(dtype=x.dtype)

        y = self.original_mlp(x)
        self.context.record(self.name, self.layer_index, x, y, timestep)

        if self.context.mode == "probe" and self.transcoder is not None:
            with torch.no_grad():
                _y_hat, latent, preactivation = self.transcoder(x, timestep, return_preactivation=True)
                self.context.record_trace(self.name, self.layer_index, preactivation, latent, timestep)
                self.context.record_latent(self.name, self.layer_index, latent, timestep)

        return y


def _core_pi05_model(policy_or_model: nn.Module) -> nn.Module:
    return getattr(policy_or_model, "model", policy_or_model)


def _module_parent(root: nn.Module, module_name: str) -> tuple[nn.Module, str]:
    parts = module_name.split(".")
    parent = root
    for part in parts[:-1]:
        parent = getattr(parent, part)
    return parent, parts[-1]


def _target_mlps(model: nn.Module) -> list[tuple[str, int, nn.Module]]:
    targets: list[tuple[str, int, nn.Module]] = []
    for name, module in model.named_modules():
        if isinstance(module, WrappedActionExpertMLP):
            continue
        match = ACTION_EXPERT_MLP_RE.fullmatch(name)
        if match is None:
            continue
        targets.append((name, int(match.group(1)), module))
    return targets


def infer_mlp_d_model(module: nn.Module) -> int:
    """Infer the hidden width of a Gemma-style MLP module."""
    for attr in ("down_proj", "up_proj", "gate_proj"):
        projection = getattr(module, attr, None)
        if isinstance(projection, nn.Linear):
            if attr == "down_proj":
                return projection.out_features
            return projection.in_features
    raise ValueError(f"Could not infer d_model for MLP module {module.__class__.__name__}")


def list_pi05_action_expert_mlp_targets(policy_or_model: nn.Module) -> list[ActionExpertMLPTarget]:
    """List action-expert MLPs targeted by DifFRACT-style transcoders."""
    model = _core_pi05_model(policy_or_model)
    return [
        ActionExpertMLPTarget(
            name=name,
            layer_index=layer_index,
            module=module,
            d_model=infer_mlp_d_model(module),
        )
        for name, layer_index, module in _target_mlps(model)
    ]


def _patch_timestep_methods(model: nn.Module, context: Pi05TranscoderContext) -> None:
    if not hasattr(model, "_pi05_mi_original_forward"):
        model._pi05_mi_original_forward = model.forward  # type: ignore[attr-defined]
    if not hasattr(model, "_pi05_mi_original_denoise_step"):
        model._pi05_mi_original_denoise_step = model.denoise_step  # type: ignore[attr-defined]
    if not hasattr(model, "_pi05_mi_original_sample_noise"):
        model._pi05_mi_original_sample_noise = model.sample_noise  # type: ignore[attr-defined]

    original_forward = model._pi05_mi_original_forward  # type: ignore[attr-defined]
    original_denoise_step = model._pi05_mi_original_denoise_step  # type: ignore[attr-defined]
    original_sample_noise = model._pi05_mi_original_sample_noise  # type: ignore[attr-defined]

    def forward_with_timestep(_self: nn.Module, *args, **kwargs):
        timestep = kwargs.get("time")
        if timestep is None:
            if len(args) < 7:
                raise TypeError("PI05Pytorch.forward wrapper could not find positional argument `time`")
            timestep = args[6]
        with context.use_timestep(timestep):
            return original_forward(*args, **kwargs)

    def denoise_step_with_timestep(_self: nn.Module, *args, **kwargs):
        timestep = kwargs.get("timestep")
        if timestep is None:
            if len(args) < 4:
                raise TypeError("PI05Pytorch.denoise_step wrapper could not find positional argument `timestep`")
            timestep = args[3]
        x_t = kwargs.get("x_t")
        if x_t is None:
            if len(args) < 3:
                raise TypeError("PI05Pytorch.denoise_step wrapper could not find positional argument `x_t`")
            x_t = args[2]
        context.record_diffusion("denoise_input", x_t, timestep)
        with context.use_timestep(timestep):
            velocity = original_denoise_step(*args, **kwargs)
        context.record_diffusion("denoise_velocity", velocity, timestep)
        return velocity

    def sample_noise_with_trace(_self: nn.Module, *args, **kwargs):
        noise = original_sample_noise(*args, **kwargs)
        context.record_diffusion("initial_noise", noise, None)
        return noise

    model.forward = types.MethodType(forward_with_timestep, model)  # type: ignore[method-assign]
    model.denoise_step = types.MethodType(denoise_step_with_timestep, model)  # type: ignore[method-assign]
    model.sample_noise = types.MethodType(sample_noise_with_trace, model)  # type: ignore[method-assign]
    model._pi05_mi_timestep_context = context  # type: ignore[attr-defined]


def install_pi05_action_expert_wrappers(
    policy_or_model: nn.Module,
    *,
    context: Pi05TranscoderContext | None = None,
    transcoders: dict[str, TimeConditionedTranscoder] | None = None,
    mode: TranscoderMode = "train",
) -> tuple[Pi05TranscoderContext, list[str]]:
    """Replace Pi0.5 action-expert MLP modules with wrapper modules.

    Args:
        policy_or_model: Either a ``PI05Policy`` or its underlying ``PI05Pytorch`` model.
        context: Optional shared context. A new one is created when omitted.
        transcoders: Optional mapping from target MLP module name to transcoder.
        mode: Initial context mode.

    Returns:
        The shared context and the list of wrapped module names.
    """
    model = _core_pi05_model(policy_or_model)
    if context is None:
        context = Pi05TranscoderContext(mode=mode)
    else:
        context.mode = mode
    transcoders = transcoders or {}

    wrapped_names: list[str] = []
    for name, layer_index, module in _target_mlps(model):
        parent, attr = _module_parent(model, name)
        setattr(
            parent,
            attr,
            WrappedActionExpertMLP(
                name=name,
                layer_index=layer_index,
                original_mlp=module,
                context=context,
                transcoder=transcoders.get(name),
            ),
        )
        wrapped_names.append(name)

    _patch_timestep_methods(model, context)
    return context, wrapped_names


In [ ]:
# @title Definitions 2 — Streaming Statistics And Checkpoint Evidence
"""Checkpoint and streaming fidelity diagnostics; no LeRobot or simulator imports."""

from dataclasses import asdict, dataclass
import hashlib
import json
from pathlib import Path
import numpy as np
import torch



@dataclass(frozen=True)
class AuditConfig:
    tasks: tuple[int, ...] = (0, 7, 8, 9)
    starts: tuple[int, ...] = (0, 1, 2)
    seeds_per_start: int = 1
    inference_steps: int = 10
    replacement_layers: tuple[int, ...] = (0, 8, 17)
    replace_all: bool = True
    action_prefix: int = 10
    seed: int = 20261112

    def cases(self):
        if (not self.tasks or not self.starts or not self.replacement_layers or
            len(set(self.tasks)) != len(self.tasks) or len(set(self.starts)) != len(self.starts) or
            len(set(self.replacement_layers)) != len(self.replacement_layers) or
            min((*self.tasks, *self.starts, *self.replacement_layers)) < 0 or
            min(self.seeds_per_start, self.inference_steps, self.action_prefix) < 1):
            raise ValueError('Audit IDs must be unique/nonnegative; counts must be positive')
        return [dict(case_id=f'task{t}_init{i}_noise{n}', task_id=t, init_state_id=i,
                     noise_index=n, scene_seed=self.seed+1009*t+i,
                     noise_seed=self.seed+100003*t+101*i+n)
                for t in self.tasks for i in self.starts for n in range(self.seeds_per_start)]


def audit_json(path, payload):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    encoded = json.dumps(payload, indent=2, sort_keys=True, allow_nan=False)
    temporary = path.with_suffix(path.suffix+'.tmp')
    temporary.write_text(encoded+'\n')
    temporary.replace(path)


def file_sha256(path):
    digest = hashlib.sha256()
    with Path(path).open('rb') as handle:
        for block in iter(lambda: handle.read(8*1024*1024), b''):
            digest.update(block)
    return digest.hexdigest()


def tensor_sha256(value):
    a = value.detach().float().cpu().contiguous().numpy()
    return hashlib.sha256(str(a.shape).encode()+a.tobytes()).hexdigest()


def batch_sha256(batch):
    digest=hashlib.sha256()
    def visit(value):
        if isinstance(value,torch.Tensor):
            digest.update(str(value.dtype).encode()); digest.update(tensor_sha256(value).encode())
        elif isinstance(value,dict):
            for key in sorted(value):digest.update(str(key).encode());visit(value[key])
        elif isinstance(value,(list,tuple)):
            digest.update(str(len(value)).encode())
            for item in value:visit(item)
        elif isinstance(value,np.ndarray):
            digest.update(str((value.shape,value.dtype)).encode());digest.update(value.tobytes())
        else:digest.update(repr(value).encode())
    visit(batch)
    return digest.hexdigest()


def checkpoint_evidence(checkpoint):
    """Inspect optimizer evidence without treating checkpoint names as updates."""
    configs, states = checkpoint['configs'], checkpoint['state_dicts']
    if set(configs) != set(states) or not configs:
        raise ValueError('Checkpoint configurations and dictionaries do not match')
    layers = []
    for name, cfg in configs.items():
        weights = states[name]
        bad = [key for key, value in weights.items() if not bool(torch.isfinite(value).all())]
        decoder = weights['decoder.weight'].float()
        norms = decoder.norm(dim=0)
        finite_number=lambda v:float(v) if bool(torch.isfinite(v)) else None
        layers.append(dict(name=name, layer=int(name.split('.')[-2]), config=cfg,
            finite=not bad, nonfinite_tensors=bad, parameters=sum(v.numel() for v in weights.values()),
            decoder_norm_min=finite_number(norms.min()), decoder_norm_mean=finite_number(norms.mean()),
            decoder_norm_max=finite_number(norms.max())))
    optimizer = checkpoint.get('optimizer_state_dict')
    parameter_rows = []
    groups = optimizer.get('param_groups', []) if optimizer else []
    ids = [p for group in groups for p in group['params']]
    for pid in ids:
        state = optimizer.get('state', {}).get(pid, {})
        step = state.get('step')
        step = float(step) if step is not None else None
        if step is not None and not np.isfinite(step):
            step = None
        parameter_rows.append(dict(parameter_id=pid, step=step))
    steps = [r['step'] for r in parameter_rows if r['step'] is not None]
    if optimizer is None:
        update_status = 'unknown_optimizer_not_saved'
    elif not optimizer.get('state'):
        update_status = 'no_optimizer_updates_recorded'
    elif ids and len(steps) == len(ids) and min(steps) > 0:
        update_status = 'positive_steps_for_all_saved_optimizer_parameters'
    else:
        update_status = 'incomplete_or_zero_step_evidence'
    history = checkpoint.get('metrics_history', [])
    train_rows = [r for r in history if 'event' not in r and 'minibatches' in r]
    bad_metrics=[dict(row=i,key=k,value=str(v)) for i,row in enumerate(history)
                 for k,v in row.items() if isinstance(v,float) and not np.isfinite(v)]
    last_metrics=[{k:(str(v) if isinstance(v,float) and not np.isfinite(v) else v)
                  for k,v in row.items()} for row in history[-5:]]
    return dict(collection_mode=checkpoint.get('collection_mode'),
        collection_feed_forwards=checkpoint.get('feed_forward'),
        policy_path=checkpoint.get('policy_path'), num_inference_steps=checkpoint.get('num_inference_steps'),
        lambda_l1=checkpoint.get('lambda_l1'), layers=layers,
        all_dictionary_weights_finite=all(r['finite'] for r in layers),
        optimizer_update_status=update_status, optimizer_parameter_count=len(ids),
        optimizer_positive_step_parameters=sum(s is not None and s > 0 for s in [r['step'] for r in parameter_rows]),
        optimizer_step_min=min(steps) if steps else None, optimizer_step_max=max(steps) if steps else None,
        optimizer_steps=parameter_rows,
        history_rows=len(history), training_rows=len(train_rows),
        history_total_layer_minibatches=sum(r.get('minibatches') or 0 for r in train_rows
                                           if r.get('minibatches') is None or np.isfinite(r['minibatches'])),
        history_rows_without_updates=sum(not r.get('minibatches') for r in train_rows),
        last_metrics=last_metrics, nonfinite_history_metrics=bad_metrics,
        limitations=['Optimizer evidence is not convergence or semantic validity.',
                    'Optimizer IDs are not assigned to layer names without an original parameter-order manifest.',
                    'Collection step numbers are not optimizer step counts.',
                    'No training split, teacher weight hash or runtime version is inferred from the filename.'])


class FitBucket:
    """Mergeable sufficient statistics: all records counted, no activation ring."""
    def __init__(self):
        self.n = 0
        self.mean = None
        self.m2 = self.sse = self.energy = self.l0 = 0.0
        self.x_energy = self.max_abs_x = self.max_abs_y = 0.0
        self.fp16_overflow_elements = 0
        self.calls = 0
        self.t_min, self.t_max = None, None

    def update(self, x, y, predicted, z, timestep):
        arrays = [v.detach().float().reshape(-1, v.shape[-1]) for v in (x, y, predicted, z)]
        if any(not bool(torch.isfinite(v).all()) for v in arrays):
            raise ValueError('Non-finite activation, reconstruction or sparse code; refusing a fidelity score')
        xx, yy, pp, zz = arrays
        if xx.shape != yy.shape or yy.shape != pp.shape or zz.shape[0] != yy.shape[0]:
            raise ValueError('Unmatched activation records')
        yd = yy.double()
        count, mean = len(yy), yd.mean(0).cpu().numpy()
        batch_m2 = float(((yd-yd.mean(0))**2).sum())
        self.merge_moments(count, mean, batch_m2)
        self.sse += float(((pp.double()-yd)**2).sum())
        self.energy += float(yd.square().sum())
        self.x_energy += float(xx.double().square().sum())
        self.l0 += float((zz > 0).sum())
        self.max_abs_x = max(self.max_abs_x, float(xx.abs().max()))
        self.max_abs_y = max(self.max_abs_y, float(yy.abs().max()))
        self.fp16_overflow_elements += int((xx.abs()>65504).sum())+int((yy.abs()>65504).sum())
        self.calls += 1
        self.t_min = timestep if self.t_min is None else min(self.t_min, timestep)
        self.t_max = timestep if self.t_max is None else max(self.t_max, timestep)
        return (zz > 0).sum(0).cpu().numpy().astype(np.int64)

    def merge_moments(self, n, mean, m2):
        if not self.n:
            self.n, self.mean, self.m2 = n, mean.copy(), m2
            return
        delta = mean-self.mean
        total = self.n+n
        self.m2 += m2+float(delta @ delta)*self.n*n/total
        self.mean += delta*n/total
        self.n = total

    def merge(self, other):
        if not other.n:
            return
        self.merge_moments(other.n, other.mean, other.m2)
        for key in ('sse', 'energy', 'l0', 'x_energy', 'fp16_overflow_elements', 'calls'):
            setattr(self, key, getattr(self,key)+getattr(other,key))
        for key in ('max_abs_x', 'max_abs_y'):
            setattr(self,key,max(getattr(self,key),getattr(other,key)))
        self.t_min = other.t_min if self.t_min is None else min(self.t_min,other.t_min)
        self.t_max = other.t_max if self.t_max is None else max(self.t_max,other.t_max)

    def row(self, common_variance=None):
        variance = self.m2/self.n if self.n else 0
        return dict(records=self.n, calls=self.calls, sse=self.sse, target_energy=self.energy,
            mse_per_vector=self.sse/self.n if self.n else None,
            own_variance_nmse=self.sse/self.m2 if self.m2>1e-12 else None,
            common_variance_nmse=self.sse/self.n/common_variance if self.n and common_variance and common_variance>1e-12 else None,
            variance_sum=variance, relative_rmse=(self.sse/self.energy)**.5 if self.energy>1e-12 else None,
            l0_mean=self.l0/self.n if self.n else None,
            input_rms=(self.x_energy/self.n/len(self.mean))**.5 if self.n else None,
            target_rms=(self.energy/self.n/len(self.mean))**.5 if self.n else None,
            max_abs_x=self.max_abs_x, max_abs_y=self.max_abs_y,
            fp16_overflow_elements=self.fp16_overflow_elements, t_min=self.t_min,t_max=self.t_max)


class AuditAccumulator:
    def __init__(self):
        self.buckets, self.occupancy, self.first_output_hashes = {}, {}, {}

    def update(self, mode, layer, x, y, predicted, z, t, call):
        # Fixed width flow-time bins, independent of sampled/trajectory call order.
        tau_bin = min(9,max(0,int(np.ceil(float(t)*10-1e-6))-1))
        key = f'{mode}|{layer}|{tau_bin}'
        bucket = self.buckets.setdefault(key,FitBucket())
        counts = bucket.update(x,y,predicted,z,float(t))
        occupancy_key = f'{mode}|{layer}'
        self.occupancy[occupancy_key] = self.occupancy.get(occupancy_key,np.zeros_like(counts))+counts
        if call == 0:
            self.first_output_hashes[occupancy_key] = tensor_sha256(y)

    def save(self, path):
        arrays = {f'occupancy::{k}':v for k,v in self.occupancy.items()}
        scalars = {}
        for key,b in self.buckets.items():
            arrays[f'mean::{key}'] = b.mean
            scalars[key] = {k:v for k,v in vars(b).items() if k!='mean'}
        arrays['metadata'] = np.array(json.dumps(dict(buckets=scalars,hashes=self.first_output_hashes)))
        temporary = Path(path).with_suffix('.tmp')
        with temporary.open('wb') as handle:
            np.savez_compressed(handle, **arrays)
        temporary.replace(path)

    @classmethod
    def load(cls, path):
        result = cls()
        with np.load(path,allow_pickle=False) as data:
            metadata=json.loads(str(data['metadata']))
            result.first_output_hashes=metadata['hashes']
            for key,values in metadata['buckets'].items():
                b=FitBucket(); b.__dict__.update(values); b.mean=data[f'mean::{key}'].copy()
                result.buckets[key]=b
            result.occupancy={k.removeprefix('occupancy::'):data[k].copy() for k in data.files if k.startswith('occupancy::')}
        return result

    def merge(self, other):
        for key,b in other.buckets.items():
            self.buckets.setdefault(key,FitBucket()).merge(b)
        for key,counts in other.occupancy.items():
            self.occupancy[key]=self.occupancy.get(key,np.zeros_like(counts))+counts


def action_error(reference, other, prefix):
    a,b=np.asarray(reference,dtype=float),np.asarray(other,dtype=float)
    if a.shape!=b.shape or a.ndim!=2 or not np.isfinite(a).all() or not np.isfinite(b).all():
        raise ValueError('Invalid or mismatched action chunks')
    def stats(a,b):
        difference=b-a; norm=np.linalg.norm(a)
        return dict(rmse=float(np.sqrt(np.mean(difference**2))),
            relative_l2=float(np.linalg.norm(difference)/norm) if norm>1e-12 else None,
            max_abs=float(np.max(np.abs(difference))))
    return dict(full=stats(a,b),prefix=stats(a[:prefix],b[:prefix]),
                prefix_steps=min(prefix,len(a)),exactly_equal=bool(np.array_equal(a,b)))


def task_interval(rows, field, *, seed=0):
    by_task={}
    for row in rows:
        value=row.get(field)
        if value is not None and np.isfinite(value):
            by_task.setdefault(row['task_id'],[]).append(value)
    means=[float(np.mean(v)) for v in by_task.values()]
    ci=None
    if len(means)>=3:
        boot=np.random.default_rng(seed).choice(means,size=(4000,len(means)),replace=True).mean(1)
        ci=np.quantile(boot,[.025,.975]).tolist()
    return dict(task_balanced_mean=float(np.mean(means)) if means else None,
                task_bootstrap_95ci=ci,tasks=len(means),observations=sum(map(len,by_task.values())))


def summarize_audit(root, config, checkpoint_report):
    root=Path(root); merged=AuditAccumulator(); cases=[]; action_rows=[]; case_fit=[]
    planned=config.cases()
    for case in planned:
        folder=root/'cases'/case['case_id']
        if not (folder/'result.json').exists():
            continue
        row=json.loads((folder/'result.json').read_text())
        if row['case']!=case:
            raise ValueError('Saved case does not match protocol')
        current=AuditAccumulator.load(folder/'fit_stats.npz')
        merged.merge(current); cases.append(row)
        per_case={}
        for key,b in current.buckets.items():
            mode,layer,_=key.split('|'); per_case.setdefault((mode,layer),FitBucket()).merge(b)
        for (mode,layer),b in per_case.items():
            if mode!='trajectory':
                native=per_case[('trajectory',layer)]
                case_fit.append(dict(task_id=case['task_id'],case_id=case['case_id'],layer=int(layer),mode=mode,
                                     mse_difference=native.sse/native.n-b.sse/b.n))
        for mode,metrics in row['actions'].items():
            action_rows.append(dict(**case,mode=mode,rmse=metrics['prefix']['rmse'],
                                    relative_l2=metrics['prefix']['relative_l2']))
    pooled={}
    for key,b in merged.buckets.items():
        mode,layer,_=key.split('|'); pooled.setdefault((mode,layer),FitBucket()).merge(b)
    fit=[]
    for (mode,layer),b in sorted(pooled.items()):
        native=pooled[('trajectory',layer)]
        counts=merged.occupancy[f'{mode}|{layer}']
        fit.append(dict(mode=mode,layer=int(layer),**b.row(native.m2/native.n),
            dictionary_width=len(counts),never_fired_in_sample=int((counts==0).sum()),
            never_fired_fraction=float((counts==0).mean()),
            occupancy_scope='Observed sample only; never-fired does not establish a dead feature'))
    per_time=[]
    for key,b in sorted(merged.buckets.items()):
        mode,layer,tau_bin=key.split('|'); native=pooled[('trajectory',layer)]
        per_time.append(dict(mode=mode,layer=int(layer),tau_bin=int(tau_bin),
                             tau_low=int(tau_bin)/10,tau_high=(int(tau_bin)+1)/10,**b.row(native.m2/native.n)))
    actions={mode:dict(relative_l2=task_interval([r for r in action_rows if r['mode']==mode],'relative_l2'),
                       rmse=task_interval([r for r in action_rows if r['mode']==mode],'rmse'))
             for mode in sorted({r['mode'] for r in action_rows})}
    contrasts=[]
    for layer in sorted({r['layer'] for r in case_fit}):
        native=pooled[('trajectory',str(layer))]; denominator=native.m2/native.n
        for mode in ('noise_grid','noise_sampled'):
            rows=[dict(r,delta=r['mse_difference']/denominator if denominator>1e-12 else None)
                  for r in case_fit if r['layer']==layer and r['mode']==mode]
            contrasts.append(dict(layer=layer,comparison='trajectory_minus_'+mode,
                                  **task_interval(rows,'delta')))
    report=dict(planned_cases=len(planned),completed_cases=len(cases),
        completed_starts=len({(r['case']['task_id'],r['case']['init_state_id']) for r in cases}),
        completed_tasks=len({r['case']['task_id'] for r in cases}),
        status='complete' if len(cases)==len(planned) else 'partial',fit=fit,per_time=per_time,actions=actions,
        reconstruction_contrasts=contrasts,
        first_step_control_failures=sum(not c['equal_within_repeat_floor'] for row in cases for c in row['first_step_controls']),
        checkpoint_update_status=checkpoint_report['optimizer_update_status'],
        limitations=['Diagnostic cohort uses initial simulator observations, not full rollouts or occlusions.',
            'States are not established as held out from policy/transcoder pretraining.',
            'Current runtime fidelity does not establish fidelity under the original training runtime.',
            'All modes use the same trajectory-derived layer variance denominator for common_variance_nmse.',
            'Relative action errors are in normalized policy action coordinates; actions are not executed.',
            'Few task clusters make bootstrap intervals imprecise; no universal fidelity threshold is assumed.',
            'No feature attribution, steerability, collision prevention or task success is certified.'])
    audit_json(root/'report.json',report)
    lines=['# Transcoder diagnostic evidence','',f"Completed **{len(cases)}/{len(planned)} cases**, {report['completed_starts']} starts, {report['completed_tasks']} tasks.",'',
           '| Layer | Inference nMSE | Fresh-noise/grid nMSE | Random-time nMSE | Inference active features/token |',
           '|---|---:|---:|---:|---:|']
    show=lambda value:'n/a' if value is None else f'{value:.5g}'
    lookup={(r['mode'],r['layer']):r for r in fit}
    for layer in sorted({r['layer'] for r in fit}):
        values=[lookup[(mode,layer)]['common_variance_nmse'] for mode in ('trajectory','noise_grid','noise_sampled')]
        lines.append('| '+str(layer)+' | '+' | '.join(map(show,values))+' | '+show(lookup[('trajectory',layer)]['l0_mean'])+' |')
    lines += ['', 'nMSE uses a common denominator from original inference activations. Lower reconstruction error is better.',
        '', '| Replacement | Prefix relative action error | 95% task-bootstrap interval | Cases |', '|---|---:|---|---:|']
    for mode,metrics in actions.items():
        value=metrics['relative_l2']; interval=value['task_bootstrap_95ci']
        lines.append(f"| {mode} | {show(value['task_balanced_mean'])} | {interval if interval else 'insufficient tasks'} | {value['observations']} |")
    lines += ['','Optimizer evidence: '+checkpoint_report['optimizer_update_status'],
              f"First-step noise-grid controls outside repeat floor: {report['first_step_control_failures']}",
              'Paired reconstruction differences and task-bootstrap intervals: report.json → reconstruction_contrasts.',
              '',*['- '+v for v in report['limitations']]]
    if report['first_step_control_failures']:
        lines += ['', '**First-step pairing control failed: treat the distribution comparison as inconclusive until resolved.**']
    (root/'summary.md').write_text('\n'.join(lines)+'\n')
    return report


In [ ]:
# @title Definitions 3 — Simulator Snapshots And Direct Audit Runner
import argparse
import logging
import os
from dataclasses import dataclass
from pathlib import Path
from typing import Any, Callable
import numpy as np
import torch

@dataclass
class Harness:
    vec_env: Any
    inner_env: Any
    prompt: str
    preprocess_observation: Callable[[dict], dict]
    # Absent on a --list-objects pass, which needs the scene but not the policy.
    policy: Any = None
    preprocessor: Any = None
    env_preprocessor: Any = None


def ensure_libero_config() -> Path | None:
    """Write LIBERO's config.yaml so importing it cannot block on input().

    ``libero/libero/__init__.py`` prompts interactively ("Do you want to specify
    a custom path for the dataset folder?") the first time it is imported if
    ``~/.libero/config.yaml`` is missing. In any non-interactive process that
    raises ``EOFError`` before a single useful line runs.

    Writing the same defaults LIBERO would have written makes the import silent.
    The keys mirror ``get_default_path_dict`` in that module.
    """
    import importlib.util

    config_dir = Path(os.environ.get("LIBERO_CONFIG_PATH") or (Path.home() / ".libero"))
    config_file = config_dir / "config.yaml"
    if config_file.exists():
        return config_file

    # find_spec on the top-level package does not execute the submodule that
    # holds the prompt, so this is safe to call before the real import.
    spec = importlib.util.find_spec("libero")
    if spec is None or not spec.origin:
        return None
    benchmark_root = Path(spec.origin).parent / "libero"

    try:
        import yaml
    except ImportError:
        return None

    payload = {
        "benchmark_root": str(benchmark_root),
        "bddl_files": str(benchmark_root / "bddl_files"),
        "init_states": str(benchmark_root / "init_files"),
        "datasets": str(benchmark_root.parent / "datasets"),
        "assets": str(benchmark_root / "assets"),
    }
    config_dir.mkdir(parents=True, exist_ok=True)
    config_file.write_text(yaml.safe_dump(payload), encoding="utf-8")
    print(f"wrote LIBERO config {config_file} (benchmark_root={benchmark_root})", flush=True)
    return config_file


def build_harness(args: argparse.Namespace, *, load_policy: bool = True) -> Harness:
    """Build the LIBERO env, and the policy stack only when it will be used.

    Listing the scene's objects needs the simulator but not Pi0.5, and loading
    the policy means a multi-GB download, so the discovery pass skips it.
    """
    ensure_libero_config()

    from lerobot.configs.policies import PreTrainedConfig
    from lerobot.envs.configs import LiberoEnv
    from lerobot.envs.factory import make_env, make_env_pre_post_processors
    from lerobot.policies.factory import make_policy, make_pre_post_processors
    from lerobot.scripts.lerobot_eval import preprocess_observation

    env_cfg = LiberoEnv(task=args.suite, task_ids=[args.task_id], fps=20)
    print(f"building LIBERO env {args.suite} task {args.task_id}", flush=True)
    envs = make_env(env_cfg, n_envs=1, use_async_envs=False)
    vec_env = envs[args.suite][args.task_id]
    inner_env = vec_env.envs[0] if hasattr(vec_env, "envs") else vec_env

    harness = Harness(
        vec_env=vec_env,
        inner_env=inner_env,
        prompt=args.prompt or "",
        preprocess_observation=preprocess_observation,
    )
    if not load_policy:
        print("skipping policy load (discovery pass)", flush=True)
        return harness

    try:
        print(f"loading policy {args.policy_path}", flush=True)
        policy_cfg = PreTrainedConfig.from_pretrained(
            args.policy_path,
            cache_dir=os.environ.get("HF_HUB_CACHE"),
            local_files_only=os.environ.get("HF_HUB_OFFLINE") == "1",
        )
        policy_cfg.pretrained_path = Path(args.policy_path)
        policy_cfg.device = args.device
        policy_cfg.dtype = args.policy_dtype
        policy_cfg.compile_model = False
        policy_cfg.gradient_checkpointing = False
        policy_cfg.n_action_steps = 10

        # LeRobot swallows a failed state-dict load and hands back random weights;
        # record what load_state_dict reported and refuse to measure on anything
        # but a complete load.
        install_load_recorder()
        policy = make_policy(cfg=policy_cfg, env_cfg=env_cfg, rename_map={})
        assert_weights_loaded(policy, source=str(args.policy_path))
        policy.eval()
        for parameter in policy.parameters():
            parameter.requires_grad_(False)

        preprocessor, _ = make_pre_post_processors(
            policy_cfg=policy_cfg,
            pretrained_path=args.policy_path,
            preprocessor_overrides={
                "device_processor": {"device": str(policy.config.device)},
                "rename_observations_processor": {"rename_map": {}},
            },
        )
        harness.policy = policy
        harness.preprocessor = preprocessor
        harness.env_preprocessor, _ = make_env_pre_post_processors(env_cfg=env_cfg, policy_cfg=policy_cfg)
        return harness
    except BaseException:
        try:
            vec_env.close()
        except Exception:
            logging.exception('LIBERO cleanup failed after policy loading failed')
        raise


def _add_batch_axis(value: Any) -> Any:
    if isinstance(value, dict):
        return {key: _add_batch_axis(item) for key, item in value.items()}
    if isinstance(value, np.ndarray):
        return value[None, ...]
    if isinstance(value, (int, float)):
        return np.asarray([value])
    return value


def _infer_prompt(harness: Harness, observation: dict[str, Any]) -> str:
    for candidate in (observation.get("task"), getattr(harness.inner_env, "task_description", None)):
        if isinstance(candidate, (list, tuple)) and candidate:
            return str(candidate[0])
        if isinstance(candidate, str) and candidate:
            return candidate
    language = getattr(getattr(harness.inner_env, "_env", None), "language_instruction", None)
    return str(language) if language else ""


"""Main-thread checkpoint audit. Embedded literally in the diagnostic notebook."""
import inspect
import hashlib
import json
import random
import time
from dataclasses import asdict
from pathlib import Path
from types import SimpleNamespace
import numpy as np
import torch



def stage_audit_checkpoint(source, cache_dir):
    """Copy once to local disk for mmap; validate the cached file on every run."""
    source, cache_dir=Path(source),Path(cache_dir)
    if not source.is_file():
        raise FileNotFoundError(f'Checkpoint not found: {source}')
    cache_dir.mkdir(parents=True,exist_ok=True)
    signature=dict(path=str(source.resolve()),size=source.stat().st_size,mtime_ns=source.stat().st_mtime_ns)
    receipt=cache_dir/'receipt.json'; local=cache_dir/'checkpoint.pt'
    old=json.loads(receipt.read_text()) if receipt.exists() else None
    if old and old['source']==signature and local.is_file() and local.stat().st_size==signature['size']:
        digest=file_sha256(local)
        if digest==old['sha256']:
            print('Reusing verified local checkpoint:',local,flush=True)
            return local,digest
    import hashlib
    digest=hashlib.sha256(); total=0; temporary=local.with_suffix('.partial')
    with source.open('rb') as reader,temporary.open('wb') as writer:
        for block in iter(lambda:reader.read(16*1024*1024),b''):
            writer.write(block); digest.update(block); total+=len(block)
            if total%(256*1024*1024)==0:
                print(f'Checkpoint copied: {total/2**30:.2f}/{signature["size"]/2**30:.2f} GiB',flush=True)
    if total!=signature['size'] or source.stat().st_mtime_ns!=signature['mtime_ns']:
        raise RuntimeError('Checkpoint changed during copy; rerun after the writer finishes')
    temporary.replace(local)
    digest=digest.hexdigest(); audit_json(receipt,dict(source=signature,sha256=digest))
    return local,digest


class AuditHooks:
    """Observe native MLPs and optionally replace them; retain exact time/context."""
    def __init__(self, policy, transcoders):
        self.policy,self.transcoders=policy,transcoders
        self.targets=list_pi05_action_expert_mlp_targets(policy)
        if set(transcoders)!={t.name for t in self.targets}:
            raise ValueError('Policy MLP names and checkpoint layers must match exactly')
        self.original=policy.model.denoise_step
        self.signature=inspect.signature(self.original)
        self.handles=[]; self.mode=None; self.acc=None; self.index=0; self.replacements=set()
        self.active=False; self.replace_calls=0; self.first_y={}; self.times=[]
        for target in self.targets:
            self.handles.append(target.module.register_forward_hook(self._hook(target)))
        def denoise(*args,**kwargs):
            bound=self.signature.bind(*args,**kwargs)
            if self.mode=='noise_grid':
                bound.arguments['x_t']=self.noise.clone()
            elif self.mode=='noise_sampled':
                bound.arguments['x_t']=self.draw_noise[self.index].clone()
                bound.arguments['timestep']=self.draw_times[self.index:self.index+1]
            self.t=float(bound.arguments['timestep'].detach().float().reshape(-1)[0])
            self.times.append(self.t); self.active=True
            try:
                return self.original(*bound.args,**bound.kwargs)
            finally:
                self.active=False; self.index+=1
        policy.model.denoise_step=denoise

    def _hook(self,target):
        def hook(module,inputs,output):
            if not self.active:
                raise RuntimeError('Action MLP called outside denoise_step; unsupported policy path')
            if self.index==0:
                self.first_y[target.layer_index]=output.detach().float().cpu().clone()
            measure=self.mode in ('trajectory','noise_grid','noise_sampled')
            replace=target.layer_index in self.replacements
            if not measure and not replace:
                return output
            tc=self.transcoders[target.name]
            t=torch.full((inputs[0].shape[0],),self.t,device=inputs[0].device,dtype=torch.float32)
            predicted,z=tc(inputs[0],t)
            if predicted.shape!=output.shape:
                raise ValueError('Transcoder output shape does not match native MLP')
            if measure:
                self.acc.update(self.mode,target.layer_index,inputs[0],output,predicted,z,self.t,self.index)
            if replace:
                self.replace_calls+=1
                if not bool(torch.isfinite(predicted).all()):
                    raise ValueError('Non-finite replacement output')
                return predicted.to(dtype=output.dtype)
            return output
        return hook

    def predict(self,batch,noise,steps,*,mode,acc,replacements=(),draw_times=None,draw_noise=None):
        before=batch_sha256(batch)
        self.mode,self.acc,self.noise=mode,acc,noise
        self.replacements=set(replacements); self.index=0; self.replace_calls=0
        self.first_y={}; self.times=[]; self.draw_times,self.draw_noise=draw_times,draw_noise
        self.policy.reset()
        with torch.inference_mode():
            action=self.policy.predict_action_chunk(batch,noise=noise.clone(),num_steps=steps)
        if self.index!=steps or self.replace_calls!=steps*len(self.replacements):
            raise RuntimeError('Denoising/replacement call count mismatch; refusing incomplete evidence')
        if batch_sha256(batch)!=before:
            raise RuntimeError('Policy mutated the shared observation batch; refusing unmatched comparisons')
        return action[0].detach().float().cpu().numpy()

    def close(self):
        self.policy.model.denoise_step=self.original
        for handle in self.handles:
            handle.remove()


def run_transcoder_audit(config,checkpoint_path,root,*,source_digest,runtime_report,
                         policy_path='lerobot/pi05_libero_finetuned',minutes=30,phase='all',
                         cache_dir='/content/pi05-audit-checkpoint',device='cuda'):
    """Save each complete case, resume exact protocols, and preserve native errors."""
    if phase not in ('metadata','all') or minutes<=0:
        raise ValueError('Use phase metadata/all and a positive collection budget')
    planned=config.cases(); root=Path(root); root.mkdir(parents=True,exist_ok=True)
    local,digest=stage_audit_checkpoint(checkpoint_path,cache_dir)
    checkpoint=torch.load(local,map_location='cpu',mmap=True,weights_only=True)
    old_manifest=root/'manifest.json'
    if old_manifest.exists():
        old=json.loads(old_manifest.read_text())
        if old['checkpoint_sha256']!=digest or old['source_digest']!=source_digest:
            raise ValueError('Checkpoint or notebook definitions changed; choose a new RUN_NAME')
    evidence=checkpoint_evidence(checkpoint)
    evidence.update(checkpoint_sha256=digest,source_path=str(checkpoint_path),local_path=str(local),
                    source_digest=source_digest)
    audit_json(root/'checkpoint_audit.json',evidence)
    print(json.dumps({k:evidence[k] for k in ('collection_mode','collection_feed_forwards',
        'optimizer_update_status','optimizer_step_min','optimizer_step_max','all_dictionary_weights_finite')},indent=2),flush=True)
    if phase=='metadata':
        return evidence
    if not evidence['all_dictionary_weights_finite']:
        raise ValueError('Checkpoint contains non-finite dictionary weights; see checkpoint_audit.json')
    layer_ids={r['layer'] for r in evidence['layers']}
    if not set(config.replacement_layers)<=layer_ids:
        raise ValueError('Requested replacement layer is absent from checkpoint')
    manifest=dict(config=asdict(config),source_digest=source_digest,checkpoint_sha256=digest,
                  runtime=runtime_report,policy_path=str(policy_path),
                  scope='Initial simulator observations; action predictions only; no executed interventions')
    manifest=json.loads(json.dumps(manifest))
    manifest_path=root/'manifest.json'
    if manifest_path.exists() and json.loads(manifest_path.read_text())!=manifest:
        raise ValueError('Diagnostic protocol/runtime/checkpoint changed. Choose a new RUN_NAME; saved cases are preserved.')
    audit_json(manifest_path,manifest)
    if all((root/'cases'/c['case_id']/'result.json').exists() for c in planned):
        return summarize_audit(root,config,evidence)
    base=None; active=None; hooks=None
    args=SimpleNamespace(suite='libero_spatial',task_id=config.tasks[0],prompt='',
                         policy_path=str(policy_path),device=device,policy_dtype='bfloat16')
    try:
        base=build_harness(args); policy=base.policy
        teacher_report=assert_weights_loaded(policy,source=str(policy_path))
        audit_json(root/'teacher_load.json',teacher_report)
        base.vec_env.close()
        transcoders={}
        for name,cfg in checkpoint['configs'].items():
            tc=TimeConditionedTranscoder(TimeConditionedTranscoderConfig(**cfg))
            tc.load_state_dict(checkpoint['state_dicts'][name],strict=True)
            tc.to(device=device,dtype=torch.float32).eval()
            for parameter in tc.parameters(): parameter.requires_grad_(False)
            transcoders[name]=tc
        del checkpoint
        hooks=AuditHooks(policy,transcoders)
        deadline=time.monotonic()+minutes*60
        for case in planned:
            folder=root/'cases'/case['case_id']
            if (folder/'result.json').exists():
                print('Reusing',case['case_id'],flush=True); continue
            if time.monotonic()>=deadline:
                print('Collection budget reached between cases; rerun to continue.',flush=True); break
            print('Auditing',case['case_id'],flush=True)
            random.seed(case['scene_seed']); np.random.seed(case['scene_seed'])
            args.task_id=case['task_id']; active=build_harness(args,load_policy=False)
            inner=active.inner_env
            if inner._init_states is None or case['init_state_id']>=len(inner._init_states):
                raise ValueError('Requested initial state unavailable; no modulo wrapping')
            inner.init_state_id=case['init_state_id']
            observation,_=inner.reset(seed=case['scene_seed'])
            prompt=_infer_prompt(active,observation)
            if not prompt: raise ValueError('No task instruction found')
            obs=base.preprocess_observation(_add_batch_axis(observation)); obs['task']=[prompt]
            batch=base.preprocessor(base.env_preprocessor(obs))
            generator=torch.Generator(device=device).manual_seed(case['noise_seed'])
            shape=(1,policy.model.config.chunk_size,policy.model.config.max_action_dim)
            noise=torch.randn(shape,generator=generator,device=device,dtype=torch.float32)
            draw_noise=torch.randn((config.inference_steps,*shape),generator=generator,device=device,dtype=torch.float32)
            # Model.sample_time does not accept a generator; isolate its global RNG use.
            with torch.random.fork_rng(devices=[torch.device(device).index or 0] if str(device).startswith('cuda') else []):
                torch.manual_seed(case['noise_seed']+1)
                draw_times=policy.model.sample_time(config.inference_steps,torch.device(device))
            acc=AuditAccumulator(); actions={}; arrays={}; first={}; time_draws={}
            for mode in ('trajectory','baseline_repeat','noise_grid','noise_sampled'):
                arrays[mode]=hooks.predict(batch,noise,config.inference_steps,mode=mode,acc=acc,
                                          draw_times=draw_times,draw_noise=draw_noise)
                first[mode]=dict(hooks.first_y); time_draws[mode]=list(hooks.times)
            actions['baseline_repeat']=action_error(arrays['trajectory'],arrays['baseline_repeat'],config.action_prefix)
            controls=[]
            for layer in sorted(layer_ids):
                native=first['trajectory'][layer]
                repeat=float((native-first['baseline_repeat'][layer]).abs().max())
                grid=float((native-first['noise_grid'][layer]).abs().max())
                controls.append(dict(layer=layer,repeat_max_abs=repeat,noise_grid_first_max_abs=grid,
                                     equal_within_repeat_floor=grid<=max(1e-6,repeat*5)))
            modes={f'layer_{layer}':[layer] for layer in config.replacement_layers}
            if config.replace_all: modes['all_layers']=sorted(layer_ids)
            for mode,layers in modes.items():
                arrays[mode]=hooks.predict(batch,noise,config.inference_steps,mode='replacement',acc=acc,replacements=layers)
                actions[mode]=action_error(arrays['trajectory'],arrays[mode],config.action_prefix)
            for layer in layer_ids:
                for mode in ('trajectory','noise_grid','noise_sampled'):
                    buckets=[b for k,b in acc.buckets.items() if k.startswith(f'{mode}|{layer}|')]
                    if sum(b.calls for b in buckets)!=config.inference_steps or sum(b.n for b in buckets)!=shape[1]*config.inference_steps:
                        raise RuntimeError('Missing activation records; refusing truncated statistics')
            folder.mkdir(parents=True,exist_ok=True)
            acc.save(folder/'fit_stats.npz')
            np.savez_compressed(folder/'actions.npz',**arrays)
            np.savez_compressed(folder/'initial_images.npz',**observation['pixels'])
            image_hashes={k:hashlib.sha256(np.asarray(v).tobytes()).hexdigest() for k,v in observation['pixels'].items()}
            # Readback verification precedes the result.json completion marker.
            loaded=AuditAccumulator.load(folder/'fit_stats.npz')
            if sum(b.n for b in loaded.buckets.values())!=sum(b.n for b in acc.buckets.values()):
                raise RuntimeError('Saved activation counts differ')
            audit_json(folder/'result.json',dict(case=case,prompt=prompt,initial_image_sha256=image_hashes,
                noise_sha256=tensor_sha256(noise),batch_sha256=batch_sha256(batch),time_draws=time_draws,actions=actions,
                first_step_controls=controls,grid_initial_time=time_draws['noise_grid'][0],
                native_initial_time=time_draws['trajectory'][0]))
            active.vec_env.close(); active=None
            completed=sum((root/'cases'/c['case_id']/'result.json').exists() for c in planned)
            audit_json(root/'state.json',dict(status='collecting',completed=completed,planned=len(planned)))
            print(f'Saved {completed}/{len(planned)} cases',flush=True)
        report=summarize_audit(root,config,evidence)
        audit_json(root/'state.json',dict(status=report['status'],completed=report['completed_cases'],planned=len(planned)))
        if (root/'failure.json').exists(): (root/'failure.json').unlink()
        return report
    except BaseException as exc:
        audit_json(root/'failure.json',dict(error_type=type(exc).__name__,message=str(exc)))
        raise
    finally:
        if hooks is not None: hooks.close()
        if active is not None:
            try: active.vec_env.close()
            except Exception: pass
        if base is not None and hooks is None:
            try: base.vec_env.close()
            except Exception: pass

AUDIT_NOTEBOOK_VERSION = 'b02ec246822dd828a5b0441673cbf8103fa235cb2865c78f5610093d3876274a'


In [ ]:
# @title Definitions 4 — Matched Fits And Decision Report
"""Controlled, small-budget training-method comparison; no steering claims."""

from dataclasses import asdict, dataclass
import hashlib
import json
import math
from pathlib import Path
import time

import numpy as np
import torch
from torch import nn



ARMS = ('A_noise_sampled', 'B_noise_grid', 'C_trajectory', 'D_paper_adapted')


@dataclass(frozen=True)
class MethodConfig:
    layer: int = 8
    train_tasks: tuple[int, ...] = (0, 1, 2, 3, 4)
    validation_tasks: tuple[int, ...] = (5, 6)
    test_tasks: tuple[int, ...] = (7, 8, 9)
    train_starts: tuple[int, ...] = (0, 1, 2, 3)
    eval_starts: tuple[int, ...] = (0, 1, 2)
    noise_draws: int = 2
    fit_seeds: tuple[int, ...] = (41, 42, 43)
    inference_steps: int = 10
    action_prefix: int = 10
    updates: int = 1000
    batch_size: int = 512
    checkpoint_every: int = 250
    lambda_l1: float = 1e-4
    seed: int = 20261112

    def cases(self):
        groups = (self.train_tasks, self.validation_tasks, self.test_tasks)
        if any(not g or len(set(g)) != len(g) for g in groups):
            raise ValueError('Each task split must be nonempty and unique')
        if any(set(groups[i]) & set(groups[j]) for i in range(3) for j in range(i)):
            raise ValueError('Task splits overlap')
        if (self.layer < 0 or self.lambda_l1 < 0 or
            not self.fit_seeds or len(set(self.fit_seeds)) != len(self.fit_seeds) or
            min(self.noise_draws, self.inference_steps, self.action_prefix, self.updates,
                self.batch_size, self.checkpoint_every) < 1 or self.batch_size < 2):
            raise ValueError('Invalid experiment counts or regularization')
        result = []
        for split, tasks in zip(('train', 'validation', 'test'), groups):
            starts = self.train_starts if split == 'train' else self.eval_starts
            if not starts or min((*starts, *tasks)) < 0 or len(set(starts)) != len(starts):
                raise ValueError('Invalid initial states or task IDs')
            for task in tasks:
                for start in starts:
                    for noise in range(self.noise_draws):
                        result.append(dict(case_id=f'task{task}_init{start}_noise{noise}', split=split,
                            task_id=task, init_state_id=start, noise_index=noise,
                            scene_seed=self.seed + 1009*task + start,
                            noise_seed=self.seed + 100003*task + 101*start + noise))
        return result


class PaperAdaptedTranscoder(TimeConditionedTranscoder):
    """Appendix E initialization/time network; VLA embedding and lambda remain adaptations."""
    def __init__(self, config):
        super().__init__(config)
        d = config.time_dim
        self.time_mlp = nn.Sequential(nn.Linear(d, d), nn.SiLU(), nn.Linear(d, d),
                                      nn.SiLU(), nn.Linear(d, 2*config.d_model))
        for module in self.time_mlp:
            if isinstance(module, nn.Linear):
                nn.init.kaiming_normal_(module.weight)
                nn.init.zeros_(module.bias)
        nn.init.zeros_(self.time_mlp[-1].weight)
        with torch.no_grad():
            self.encoder.weight.copy_(self.decoder.weight.T)
        self.renormalize_decoder_columns()


def method_model(base_config, arm, seed):
    if arm not in ARMS:
        raise ValueError(arm)
    cfg = dict(base_config)
    if arm == ARMS[3]:
        cfg.update(time_embedding_dim=256, time_hidden_dim=256, eps=1e-6)
    # CPU construction is paired and does not change teacher CUDA RNG state.
    with torch.random.fork_rng(devices=[]):
        torch.default_generator.manual_seed(seed)
        cls = PaperAdaptedTranscoder if arm == ARMS[3] else TimeConditionedTranscoder
        return cls(TimeConditionedTranscoderConfig(**cfg))


def method_loss(model, x, t, y, common_variance, arm, lambda_l1):
    prediction, latent = model(x, t)
    if arm == ARMS[3]:
        denominator = y.float().var(0, unbiased=False).sum() + 1e-6
    else:
        denominator = torch.as_tensor(common_variance, device=y.device).clamp_min(1e-8)
    mse = (prediction.float()-y.float()).square().sum(-1).mean()/denominator
    l1 = latent.float().abs().sum(-1).mean()
    return mse + lambda_l1*l1, mse, (latent > 0).float().sum(-1).mean()


def atomic_torch_save(payload, path):
    path = Path(path); path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_suffix('.partial')
    torch.save(payload, tmp); tmp.replace(path)


def records_for(root, cases, split, mode):
    """All raw records, float32 on CPU; no ring truncation or stochastic selection."""
    chunks = []
    for case in cases:
        if case['split'] != split:
            continue
        folder = Path(root)/'cases'/case['case_id']
        receipt = json.loads((folder/'collected.json').read_text())
        path = folder/(mode+'.npz')
        if file_sha256(path) != receipt['files'][path.name]:
            raise ValueError(f'Activation file changed: {path}')
        with np.load(path) as data:
            chunks.append({k: torch.from_numpy(data[k].copy()) for k in ('x', 'y', 't')})
    if not chunks:
        raise ValueError(f'No {split} records')
    return {k: torch.cat([c[k] for c in chunks]) for k in chunks[0]}


@torch.no_grad()
def reconstruction_metrics(model, records, denominator, batch_size=512):
    device = next(model.parameters()).device
    sse = l0 = 0.; n = len(records['t']); per_time = {}
    for offset in range(0, n, batch_size):
        x, y, t = (records[k][offset:offset+batch_size].to(device) for k in ('x', 'y', 't'))
        prediction, z = model(x, t)
        if not torch.isfinite(prediction).all() or not torch.isfinite(z).all():
            raise ValueError('Nonfinite dictionary evaluation')
        errors = (prediction.double()-y.double()).square().sum(-1)
        sse += float(errors.sum()); l0 += float((z > 0).sum())
        for value in t.unique():
            key = f'{float(value):.6f}'; mask = t == value
            bucket = per_time.setdefault(key, [0., 0])
            bucket[0] += float(errors[mask].sum()); bucket[1] += int(mask.sum())
    return dict(nmse=sse/n/denominator, mean_l0=l0/n, records=n,
                per_time={k: dict(nmse=v[0]/v[1]/denominator, records=v[1]) for k,v in per_time.items()})


def fit_method(root, config, base_config, arm, seed, train, validation, denominator,
               deadline, device='cuda'):
    """Atomic checkpoint boundaries; resume RNG/Adam state, fixed final-step comparison."""
    folder = Path(root)/'fits'/f'{arm}_seed{seed}'; folder.mkdir(parents=True, exist_ok=True)
    done = folder/'complete.json'
    if done.exists():
        receipt = json.loads(done.read_text())
        if file_sha256(folder/'model.pt') != receipt['sha256']:
            raise ValueError('Completed model checksum changed')
        return True
    model = method_model(base_config, arm, seed).to(device)
    paper = arm == ARMS[3]
    optimizer = torch.optim.AdamW(model.parameters(), lr=2e-4 if paper else 1e-4,
                                 weight_decay=0. if paper else .01)
    generator = torch.Generator().manual_seed(seed + 90000)
    state_path = folder/'resume.pt'; start = 0; history = []
    if state_path.exists():
        saved = torch.load(state_path, map_location='cpu', weights_only=True)
        if saved['arm'] != arm or saved['seed'] != seed:
            raise ValueError('Wrong fit resume identity')
        model.load_state_dict(saved['model'], strict=True)
        optimizer.load_state_dict(saved['optimizer'])
        generator.set_state(saved['sampler_state']); start = saved['step']; history = saved['history']
    elif time.monotonic() < deadline:
        history.append(dict(step=0, validation=reconstruction_metrics(model, validation, denominator)))
    def save_progress(step):
        atomic_torch_save(dict(arm=arm, seed=seed, step=step, model=model.state_dict(),
            optimizer=optimizer.state_dict(), sampler_state=generator.get_state(), history=history), state_path)
        audit_json(folder/'learning_curve.json', history)
    step = start
    try:
        while step < config.updates:
            if time.monotonic() >= deadline:
                save_progress(step); return False
            indices = torch.randint(len(train['t']), (config.batch_size,), generator=generator)
            x, y, t = (train[k][indices].to(device) for k in ('x', 'y', 't'))
            if paper:
                optimizer.param_groups[0]['lr'] = 2e-4*.5*(1+math.cos(math.pi*step/config.updates))
            optimizer.zero_grad(set_to_none=True)
            loss, mse, l0 = method_loss(model, x, t, y, denominator, arm, config.lambda_l1)
            if not torch.isfinite(loss):
                raise ValueError('Nonfinite training loss')
            loss.backward()
            grad_norm = torch.nn.utils.clip_grad_norm_(model.parameters(), 1., error_if_nonfinite=True)
            optimizer.step(); model.renormalize_decoder_columns(); step += 1
            if step % config.checkpoint_every == 0 or step == config.updates:
                history.append(dict(step=step, train_loss=float(loss.detach()), train_nmse=float(mse.detach()),
                    train_l0=float(l0), grad_norm=float(grad_norm), lr=optimizer.param_groups[0]['lr'],
                    validation=reconstruction_metrics(model, validation, denominator)))
                save_progress(step)
                print(f'{arm} seed={seed}: {step}/{config.updates}, validation nMSE={history[-1]["validation"]["nmse"]:.5f}', flush=True)
        atomic_torch_save(dict(config=asdict(model.config), state_dict=model.state_dict(), arm=arm,
                              seed=seed, step=step), folder/'model.pt')
        audit_json(done, dict(sha256=file_sha256(folder/'model.pt'), updates=step,
            records_presented=step*config.batch_size, unique_available_records=len(train['t']), history=history))
        state_path.unlink(missing_ok=True)
        return True
    finally:
        del model, optimizer
        if str(device).startswith('cuda'): torch.cuda.empty_cache()


def paired_method_interval(rows, left, right, metric, fit_seeds):
    """Average paired fit/noise repeats within task; bootstrap tasks, not tokens."""
    index = {(r['model'], r['case']['case_id']): r for r in rows}
    case_ids = sorted({r['case']['case_id'] for r in rows})
    pairs = []
    for case_id in case_ids:
        for seed in fit_seeds:
            l, r = index[(f'{left}_seed{seed}', case_id)], index[(f'{right}_seed{seed}', case_id)]
            if l[metric] is not None and r[metric] is not None:
                pairs.append(dict(task_id=l['case']['task_id'], seed=seed, delta=l[metric]-r[metric]))
    result = task_interval(pairs, 'delta')
    result['excluded_undefined_pairs'] = len(case_ids)*len(fit_seeds)-len(pairs)
    result['per_seed_mean'] = {str(s): task_interval([p for p in pairs if p['seed']==s], 'delta')['task_balanced_mean'] for s in fit_seeds}
    return result


def method_report(root, config):
    root = Path(root); cases = config.cases(); test = [c for c in cases if c['split']=='test']
    names = ['existing']+[f'{a}_seed{s}' for s in config.fit_seeds for a in ARMS]
    rows = []
    for case in test:
        for name in names:
            path = root/'evaluation'/case['case_id']/(name+'.json')
            if path.exists(): rows.append(json.loads(path.read_text()))
    complete = len(rows) == len(test)*len(names)
    report = dict(status='complete' if complete else 'partial', completed_model_cases=len(rows),
        planned_model_cases=len(test)*len(names), test_tasks=len(config.test_tasks),
        test_starts=len(config.test_tasks)*len(config.eval_starts), noise_draws_per_start=config.noise_draws,
        fit_seeds=list(config.fit_seeds), training_updates=config.updates,
        records_presented_per_fit=config.updates*config.batch_size, comparisons={}, models={}, controls={}, training_progress={},
        limitations=[
            'Small-budget layer-specific adaptation, not a full DifFRACT reproduction or historical training replay.',
            'A/B/C share the original architecture/init/Adam settings but use a fixed shared normalizer and cached uniform batches.',
            'All new activation records use float32, unlike the original trainer buffer default float16; new-arm contrasts hold this fixed.',
            'D changes a bundle of time architecture, initialization, optimization and normalization; individual causes are not isolated.',
            'D keeps VLA-specific sinusoidal periods, lambda and clipping; smaller batches/data/budget than the paper.',
            'Original checkpoint has a different training history; fresh-arm contrasts isolate methodology, old/new contrasts do not.',
            'Reported nMSE uses the shared training-trajectory variance; do not compare numerically with an earlier audit using a different denominator.',
            'Only initial simulator observations; no actions executed, attribution validation, steering or safety claims.',
            'Task bootstrap is conditional on these fit seeds and observations; three test task clusters give weak population inference.',
            'States are held out from these new fits, not necessarily from policy or old transcoder pretraining.',
            'A null result at this budget does not establish that retraining is unnecessary; inspect learning curves and L0.',
        ])
    controls = []
    for case in test:
        path = root/'evaluation'/case['case_id']/'controls.json'
        if path.exists():
            control = json.loads(path.read_text())
            controls.append(dict(task_id=case['task_id'], **{k:control[k]['prefix']['relative_l2'] for k in ('repeat','zero','mean')}))
    report['completed_control_cases'] = len(controls)
    report['controls'] = {k:task_interval(controls,k) for k in ('repeat','zero','mean')}
    if complete and len(controls) != len(test):
        raise ValueError('Model results exist without complete sensitivity controls')
    if complete:
        for name in names[1:]:
            path = root/'fits'/name/'complete.json'
            saved = json.loads(path.read_text()); history = saved['history']
            val = [r['validation']['nmse'] for r in history]
            report['training_progress'][name] = dict(updates=saved['updates'],
                unique_available_records=saved['unique_available_records'], records_presented=saved['records_presented'],
                initial_validation_nmse=val[0],final_validation_nmse=val[-1],
                last_validation_change=val[-1]-val[-2] if len(val)>1 else None)
        for name in names:
            subset = [dict(r, task_id=r['case']['task_id']) for r in rows if r['model']==name]
            report['models'][name] = {m: task_interval(subset, m) for m in ('nmse','mean_l0','action_relative_l2')}
        for left, right in ((ARMS[1],ARMS[0]),(ARMS[2],ARMS[1]),(ARMS[3],ARMS[2])):
            report['comparisons'][left+'_minus_'+right] = {
                m: paired_method_interval(rows,left,right,m,config.fit_seeds) for m in ('nmse','action_relative_l2','mean_l0')}
        report['interpretation'] = ('Negative nMSE/action differences favor the left arm. Read together with L0, '
            'per-seed consistency, learning curves, repeat floor and zero/mean controls. '
            'Improvement supports further retraining at this layer/budget; no automatic deployment or attribution verdict.')
    else:
        report['interpretation'] = 'Incomplete matched experiment: no primary arm comparison released. Resume the same run.'
    audit_json(root/'report.json', report)
    lines = ['# Training-method decision pilot', '', report['interpretation'], '',
             f'Model-cases: {len(rows)}/{len(test)*len(names)}. Test: {report["test_starts"]} starts, '
             f'{len(config.test_tasks)} tasks, {config.noise_draws} noise draws/start.', '',
             '| Model | Inference nMSE | Active features/token | Relative action error |', '|---|---:|---:|---:|']
    fmt = lambda v: 'undefined' if v is None else f'{v:.5f}'
    for name, value in report['models'].items():
        lines.append(f'| {name} | {fmt(value["nmse"]["task_balanced_mean"])} | {fmt(value["mean_l0"]["task_balanced_mean"])} | {fmt(value["action_relative_l2"]["task_balanced_mean"])} |')
    lines += ['', '## Paired comparisons (negative error differences favor left arm)', '',
              '| Comparison | Δ nMSE | Task-bootstrap 95% interval | Δ action error |', '|---|---:|---|---:|']
    for name, values in report['comparisons'].items():
        lines.append(f'| {name} | {fmt(values["nmse"]["task_balanced_mean"])} | {values["nmse"]["task_bootstrap_95ci"]} | {fmt(values["action_relative_l2"]["task_balanced_mean"])} |')
    lines += ['', '## Layer sensitivity controls', '', '| Control | Relative action error | Cases |', '|---|---:|---:|']
    for name,value in report['controls'].items():
        lines.append(f'| {name} | {fmt(value["task_balanced_mean"])} | {value["observations"]} |')
    lines += ['', *['- '+s for s in report['limitations']]]
    (root/'summary.md').write_text('\n'.join(lines)+'\n')
    return report


In [ ]:
# @title Definitions 5 — Shared Teacher Collection And Evaluation
"""Notebook-thread runner for a controlled layer-level retraining decision."""

from dataclasses import asdict
import gc
import hashlib
import inspect
import json
from pathlib import Path
import random
import time
from types import SimpleNamespace

import numpy as np
import torch



def cpu_tree(value):
    if isinstance(value, torch.Tensor): return value.detach().cpu().clone()
    if isinstance(value, dict): return {k: cpu_tree(v) for k,v in value.items()}
    if isinstance(value, list): return [cpu_tree(v) for v in value]
    if isinstance(value, tuple): return tuple(cpu_tree(v) for v in value)
    return value


def device_tree(value, device):
    if isinstance(value, torch.Tensor): return value.to(device)
    if isinstance(value, dict): return {k: device_tree(v,device) for k,v in value.items()}
    if isinstance(value, list): return [device_tree(v,device) for v in value]
    if isinstance(value, tuple): return tuple(device_tree(v,device) for v in value)
    return value


def snapshot_evidence(policy_path):
    path = Path(policy_path)
    if not path.is_dir():
        raise ValueError('Resolve a local policy snapshot before starting the experiment')
    files = {str(p.relative_to(path)): file_sha256(p) for p in sorted(path.rglob('*'))
             if p.is_file() and (p.suffix in ('.json','.safetensors','.model') or p.name.endswith('.bin'))}
    if not files or not any(n.endswith(('.safetensors','.bin')) for n in files):
        raise ValueError('No teacher weight files in policy snapshot')
    return dict(path=str(path.resolve()), files=files,
        sha256=hashlib.sha256(json.dumps(files, sort_keys=True).encode()).hexdigest())


class MethodHooks:
    """One layer only. Capture native outputs; substitute only during fidelity evaluation."""
    def __init__(self, policy, layer):
        self.policy = policy
        self.target = next(t for t in list_pi05_action_expert_mlp_targets(policy) if t.layer_index == layer)
        self.original = policy.model.denoise_step
        self.signature = inspect.signature(self.original)
        self.active = False
        self.handle = self.target.module.register_forward_hook(self.hook)
        def denoise(*args, **kwargs):
            bound = self.signature.bind(*args, **kwargs)
            if self.mode in ('A_noise_sampled','B_noise_grid'):
                bound.arguments['x_t'] = self.noise.clone()
            if self.mode == 'A_noise_sampled':
                bound.arguments['timestep'] = self.draw_times[self.call:self.call+1]
            self.t = float(bound.arguments['timestep'].detach().float().reshape(-1)[0])
            self.active = True
            try: return self.original(*bound.args, **bound.kwargs)
            finally:
                self.active = False
                self.call += 1
        policy.model.denoise_step = denoise

    def hook(self, module, inputs, output):
        if not self.active: raise RuntimeError('MLP called outside denoising')
        self.hook_calls += 1
        if self.capture:
            x = inputs[0].detach().float().reshape(-1, inputs[0].shape[-1]).cpu().clone()
            y = output.detach().float().reshape_as(x).cpu().clone()
            if not torch.isfinite(x).all() or not torch.isfinite(y).all():
                raise ValueError('Nonfinite teacher activation')
            self.records.append(dict(x=x, y=y, t=torch.full((len(x),), self.t)))
        if self.replacement == 'zero': return torch.zeros_like(output)
        if self.replacement == 'mean': return self.mean.to(output).expand_as(output)
        if self.transcoder is not None:
            times = torch.full((inputs[0].shape[0],),self.t,device=inputs[0].device)
            predicted, _ = self.transcoder(inputs[0], times)
            if predicted.shape != output.shape or not torch.isfinite(predicted).all():
                raise ValueError('Invalid replacement output')
            return predicted.to(output.dtype)
        return output

    def predict(self, batch, noise, steps, *, mode='C_trajectory', draw_times=None,
                capture=False, transcoder=None, replacement=None, mean=None):
        before = batch_sha256(batch)
        self.mode, self.noise, self.draw_times = mode, noise, draw_times
        self.capture, self.transcoder, self.replacement, self.mean = capture, transcoder, replacement, mean
        self.records = []; self.call = self.hook_calls = 0
        self.policy.reset()
        try:
            with torch.inference_mode():
                actions = self.policy.predict_action_chunk(batch, noise=noise.clone(),num_steps=steps)
            if self.call != steps or self.hook_calls != steps:
                raise RuntimeError('Incomplete denoising or layer hook coverage')
            if before != batch_sha256(batch): raise RuntimeError('Observation batch mutated')
            records = {k:torch.cat([r[k] for r in self.records]) for k in ('x','y','t')} if capture else None
            if capture and len(records['t']) != steps*noise.shape[1]:
                raise RuntimeError('Activation record count mismatch')
            return actions[0].detach().float().cpu().numpy(), records
        finally:
            self.transcoder = None
            self.records = []

    def close(self):
        self.policy.model.denoise_step = self.original
        self.handle.remove()


def collect_method_case(root, case, config, base, hooks, args, deadline, device):
    folder = root/'cases'/case['case_id']; receipt = folder/'collected.json'
    if receipt.exists():
        saved = json.loads(receipt.read_text())
        if saved['case'] != case: raise ValueError('Case identity changed')
        for name, digest in saved['files'].items():
            if file_sha256(folder/name) != digest: raise ValueError(f'Corrupt case file: {name}')
        return True
    if time.monotonic() >= deadline: return False
    random.seed(case['scene_seed']); np.random.seed(case['scene_seed'])
    args.task_id = case['task_id']; active = build_harness(args, load_policy=False)
    try:
        inner = active.inner_env
        if inner._init_states is None or case['init_state_id'] >= len(inner._init_states):
            raise ValueError('Initial state unavailable; no modulo fallback')
        inner.init_state_id = case['init_state_id']
        observation, _ = inner.reset(seed=case['scene_seed'])
        prompt = _infer_prompt(active, observation)
        if not prompt: raise ValueError('Missing original task instruction')
        obs = base.preprocess_observation(_add_batch_axis(observation)); obs['task'] = [prompt]
        batch = base.preprocessor(base.env_preprocessor(obs))
        generator = torch.Generator(device=device).manual_seed(case['noise_seed'])
        shape = (1,base.policy.model.config.chunk_size,base.policy.model.config.max_action_dim)
        noise = torch.randn(shape, generator=generator,device=device,dtype=torch.float32)
        with torch.random.fork_rng(devices=[torch.device(device).index or 0] if str(device).startswith('cuda') else []):
            torch.manual_seed(case['noise_seed']+1)
            times = base.policy.model.sample_time(config.inference_steps, torch.device(device))
        folder.mkdir(parents=True,exist_ok=True)
        native, native_records = hooks.predict(batch,noise,config.inference_steps,capture=True)
        repeat, _ = hooks.predict(batch,noise,config.inference_steps)
        repeat_error = action_error(native,repeat,config.action_prefix)
        if not repeat_error['exactly_equal']:
            raise RuntimeError('Native repeat differs; resolve nondeterminism before matched training comparisons')
        np.savez(folder/'C_trajectory.npz',**{k:v.numpy() for k,v in native_records.items()})
        files = ['C_trajectory.npz']
        if case['split'] == 'train':
            for mode in ARMS[:2]:
                _, records = hooks.predict(batch,noise,config.inference_steps,mode=mode,draw_times=times,capture=True)
                if mode == ARMS[1] and not torch.equal(records['y'][:shape[1]],native_records['y'][:shape[1]]):
                    raise RuntimeError('First-step noise-grid/trajectory identity failed')
                np.savez(folder/(mode+'.npz'), **{k:v.numpy() for k,v in records.items()})
                files.append(mode+'.npz')
        atomic_torch_save(dict(batch=cpu_tree(batch), noise=noise.cpu(), native=torch.from_numpy(native)),folder/'input.pt')
        files.append('input.pt')
        audit_json(receipt,dict(case=case,prompt=prompt,batch_sha256=batch_sha256(batch),
            files={name:file_sha256(folder/name) for name in files},records_per_mode=len(native_records['t']),
            repeat=repeat_error))
        print('Collected',case['case_id'],case['split'],flush=True)
        return True
    finally:
        active.vec_env.close()


def evaluate_method_case(root, case, config, hooks, models, denominator, mean, deadline, device):
    folder = root/'cases'/case['case_id']; out = root/'evaluation'/case['case_id']
    out.mkdir(parents=True,exist_ok=True)
    receipt = json.loads((folder/'collected.json').read_text())
    if receipt['case'] != case: raise ValueError('Evaluation case identity changed')
    for name in ('input.pt','C_trajectory.npz'):
        if file_sha256(folder/name) != receipt['files'][name]:
            raise ValueError(f'Evaluation input changed: {name}')
    saved = torch.load(folder/'input.pt',map_location='cpu',weights_only=True)
    batch, noise = device_tree(saved['batch'],device), saved['noise'].to(device)
    if batch_sha256(batch) != receipt['batch_sha256']:
        raise ValueError('Saved preprocessed observation hash changed')
    native = saved['native'].numpy()
    with np.load(folder/'C_trajectory.npz') as arrays:
        records = {k:torch.from_numpy(arrays[k].copy()) for k in ('x','y','t')}
    if not (out/'controls.json').exists():
        if time.monotonic() >= deadline: return False
        controls = {}; predictions = dict(native=native)
        for mode in ('repeat','zero','mean'):
            predicted, _ = hooks.predict(batch,noise,config.inference_steps,
                replacement=None if mode=='repeat' else mode,mean=mean)
            controls[mode] = action_error(native,predicted,config.action_prefix)
            predictions[mode] = predicted
        if not controls['repeat']['exactly_equal']:
            raise RuntimeError('Saved observation/noise no longer reproduces original action; teacher/runtime drift')
        np.savez(out/'controls_actions.npz',**predictions)
        audit_json(out/'controls.json',dict(case=case,**controls))
    for name, loader in models:
        if (out/(name+'.json')).exists(): continue
        if time.monotonic() >= deadline: return False
        tc = loader().to(device).eval()
        try:
            fit = reconstruction_metrics(tc,records,denominator)
            predicted, _ = hooks.predict(batch,noise,config.inference_steps,transcoder=tc)
            errors = action_error(native,predicted,config.action_prefix)
            np.savez(out/(name+'_actions.npz'),native=native,replacement=predicted)
            audit_json(out/(name+'.json'),dict(case=case,model=name,**fit,
                action_relative_l2=errors['prefix']['relative_l2'],action_errors=errors))
        finally:
            del tc
            if str(device).startswith('cuda'): torch.cuda.empty_cache()
        print('Evaluated',case['case_id'],name,flush=True)
    return True


def run_methodology(config, checkpoint_path, root, *, source_digest, runtime_report, policy_path,
                    minutes=120, phase='all', device='cuda', cache_dir='/content/pi05-audit-checkpoint'):
    """Budget checked at safe boundaries; setup/loading and an in-flight operation may overrun."""
    if phase not in ('collect','train','evaluate','all') or minutes <= 0:
        raise ValueError('Invalid phase or budget')
    started = time.monotonic(); deadline = started+minutes*60
    cases = config.cases(); root = Path(root); root.mkdir(parents=True,exist_ok=True)
    local, digest = stage_audit_checkpoint(checkpoint_path,cache_dir)
    checkpoint = torch.load(local,map_location='cpu',mmap=True,weights_only=True)
    names = [n for n in checkpoint['configs'] if int(n.split('.')[-2]) == config.layer]
    if len(names) != 1: raise ValueError('Checkpoint layer missing or ambiguous')
    name = names[0]; base_config = checkpoint['configs'][name]
    def old_model():
        tc = TimeConditionedTranscoder(TimeConditionedTranscoderConfig(**base_config))
        tc.load_state_dict(checkpoint['state_dicts'][name],strict=True)
        if not all(torch.isfinite(p).all() for p in tc.parameters()): raise ValueError('Nonfinite old checkpoint')
        return tc
    manifest = dict(config=asdict(config),checkpoint_sha256=digest,source_digest=source_digest,
        runtime=runtime_report,teacher=snapshot_evidence(policy_path),base_config=base_config,
        protocol='methodology-v1; no interventions executed; fixed final update, no test-based selection')
    manifest = json.loads(json.dumps(manifest))
    path = root/'manifest.json'
    if path.exists() and json.loads(path.read_text()) != manifest:
        raise ValueError('Protocol/teacher/runtime changed. Use a new RUN_NAME; old evidence is preserved.')
    audit_json(path,manifest)
    base = hooks = None
    args = SimpleNamespace(suite='libero_spatial',task_id=config.train_tasks[0],prompt='',
                           policy_path=str(policy_path),device=device,policy_dtype='bfloat16')
    def load_teacher():
        nonlocal base, hooks
        if base is None:
            base = build_harness(args)
            audit_json(root/'teacher_load.json',assert_weights_loaded(base.policy,source=str(policy_path)))
            base.vec_env.close()
            hooks = MethodHooks(base.policy,config.layer)
    try:
        collected = all((root/'cases'/c['case_id']/'collected.json').exists() for c in cases)
        if phase in ('collect','all'):
            if not collected: load_teacher()
            for case in cases:
                if not collect_method_case(root,case,config,base,hooks,args,deadline,device):
                    return method_report(root,config)
            collected = True
        if phase == 'collect': return method_report(root,config)
        if not collected: raise RuntimeError('Complete collection before training or evaluation')
        training = records_for(root,cases,'train','C_trajectory')
        # Frozen train-only denominator shared by A/B/C and all reported evaluation metrics.
        denominator = float(training['y'].double().var(0,unbiased=False).sum())
        if not np.isfinite(denominator) or denominator <= 1e-12: raise ValueError('Degenerate training variance')
        mean = training['y'].mean(0)
        audit_json(root/'normalizer.json',dict(variance=denominator,records=len(training['t']),source='train C_trajectory'))
        validation = records_for(root,cases,'validation','C_trajectory')
        if phase in ('train','all'):
            # Free the teacher before fitting. Reload from the identical hashed snapshot for evaluation.
            if hooks is not None: hooks.close(); hooks = None
            base = None; gc.collect()
            if str(device).startswith('cuda'): torch.cuda.empty_cache()
            for arm in ARMS:
                data = training if arm in ARMS[2:] else records_for(root,cases,'train',arm)
                if len(data['t']) != len(training['t']): raise ValueError('Unequal training record budgets')
                for seed in config.fit_seeds:
                    if not fit_method(root,config,base_config,arm,seed,data,validation,denominator,deadline,device):
                        return method_report(root,config)
                if arm not in ARMS[2:]: del data
        if phase == 'train': return method_report(root,config)
        models = [('existing',old_model)]
        for seed in config.fit_seeds:
            for arm in ARMS:
                folder = root/'fits'/f'{arm}_seed{seed}'
                if not (folder/'complete.json').exists(): raise RuntimeError('Finish all matched fits before test evaluation')
                receipt = json.loads((folder/'complete.json').read_text())
                if file_sha256(folder/'model.pt') != receipt['sha256']: raise ValueError('Model checksum changed')
                def loader(path=folder/'model.pt',arm=arm,seed=seed):
                    saved = torch.load(path,map_location='cpu',weights_only=True)
                    tc = method_model(base_config,arm,seed)
                    tc.load_state_dict(saved['state_dict'],strict=True)
                    return tc
                models.append((f'{arm}_seed{seed}',loader))
        del training, validation
        if all((root/'evaluation'/c['case_id']/(n+'.json')).exists()
               for c in cases if c['split']=='test' for n,_ in models):
            return method_report(root,config)
        if time.monotonic() >= deadline: return method_report(root,config)
        load_teacher()
        # Keep one dictionary resident across cases; avoid re-reading multi-GB weights from Drive per scene.
        for model_name, loader in models:
            if time.monotonic() >= deadline: break
            tc = loader().to(device).eval()
            try:
                for case in cases:
                    if case['split']=='test' and not evaluate_method_case(root,case,config,hooks,
                            [(model_name,lambda:tc)],denominator,mean,deadline,device):
                        break
            finally:
                del tc
                if str(device).startswith('cuda'): torch.cuda.empty_cache()
        return method_report(root,config)
    except BaseException as exc:
        audit_json(root/'failure.json',dict(type=type(exc).__name__,message=str(exc)))
        raise
    finally:
        if hooks is not None: hooks.close()
        if base is not None: base.vec_env.close()
        audit_json(root/'last_session.json',dict(elapsed_seconds=time.monotonic()-started,requested_minutes=minutes))

METHOD_NOTEBOOK_VERSION = 'ff0fb2e286ad0df6cfc071d23a1450f4c5ced400d3cffe92c5566b5bbd3180aa'


In [ ]:
# @title Definitions 6 — Training Recipe, Dashboard, Durable Checkpoints
"""Resumable trajectory-based transcoder training and live notebook observability."""

from dataclasses import asdict, dataclass
import hashlib
import json
import math
from pathlib import Path
import shutil
import time
import uuid

import numpy as np
import torch



@dataclass(frozen=True)
class TrajectoryTrainConfig:
    layers: tuple[int, ...] = tuple(range(18))
    cycles: int = 256
    frames_per_cycle: int = 64
    batch_size: int = 1024
    inference_steps: int = 10
    expansion: int = 16
    lr: float = 2e-4
    lambda_l1: float = 1e-4
    grad_clip: float = 1.
    seed: int = 20261113
    validation_frames: int = 24
    test_frames: int = 16
    reference_frames: int = 16
    validate_every: int = 8
    log_every: int = 25
    action_validation_frames: int = 2

    def validate(self):
        if not self.layers or len(set(self.layers))!=len(self.layers) or min(self.layers)<0:
            raise ValueError('Layers must be unique nonnegative IDs')
        if min(self.cycles,self.frames_per_cycle,self.batch_size,self.inference_steps,self.expansion,
               self.validation_frames,self.test_frames,self.reference_frames,self.validate_every,self.log_every)<1:
            raise ValueError('Training counts must be positive')
        if self.batch_size<2 or self.lr<=0 or self.lambda_l1<0 or self.grad_clip<=0:
            raise ValueError('Invalid optimizer settings')
        if not 0<=self.action_validation_frames<=self.validation_frames:
            raise ValueError('Invalid action validation count')


def training_plan(episodes, config):
    """Episode-disjoint 80/10/10; training frames cover whole episodes without early-frame bias."""
    config.validate()
    if len(episodes)<10: raise ValueError('Need at least 10 episodes for an 80/10/10 split')
    ids=[int(e['episode_index']) for e in episodes]
    if len(set(ids))!=len(ids): raise ValueError('Duplicate episode IDs')
    ranges=sorted((int(e['dataset_from_index']),int(e['dataset_to_index'])) for e in episodes)
    if any(a<0 or b<=a for a,b in ranges) or any(b>c for (_,b),(c,_) in zip(ranges,ranges[1:])):
        raise ValueError('Invalid/overlapping episode frame ranges')
    rng=np.random.default_rng(config.seed); order=rng.permutation(len(episodes))
    nval=max(1,len(order)//10); ntest=nval
    split_indices={'train':order[:len(order)-nval-ntest],
                   'validation':order[len(order)-nval-ntest:len(order)-ntest], 'test':order[-ntest:]}
    pools={}; split_ids={}
    for split,indices in split_indices.items():
        chosen=[episodes[int(i)] for i in indices]
        split_ids[split]=sorted(int(e['episode_index']) for e in chosen)
        pools[split]=np.concatenate([np.arange(int(e['dataset_from_index']),int(e['dataset_to_index'])) for e in chosen])
    count=config.cycles*config.frames_per_cycle
    train=[]
    while len(train)<count: train.extend(rng.permutation(pools['train']).tolist())
    def fixed(split,n):
        if n>len(pools[split]): raise ValueError(f'Not enough {split} frames')
        return rng.choice(pools[split],size=n,replace=False).tolist()
    return dict(episode_ids=split_ids,available_frames={k:len(v) for k,v in pools.items()},
        train_frames=train[:count],reference_frames=fixed('train',config.reference_frames),
        validation_frames=fixed('validation',config.validation_frames),test_frames=fixed('test',config.test_frames),
        split_scope='Episode holdout, not task holdout and not held out from policy pretraining')


def new_trajectory_model(d_model, config, layer):
    with torch.random.fork_rng(devices=[]):
        torch.default_generator.manual_seed(config.seed+layer)
        return PaperAdaptedTranscoder(TimeConditionedTranscoderConfig(d_model=d_model,
            expansion_factor=config.expansion,time_embedding_dim=256,time_hidden_dim=256,eps=1e-6))


def load_trajectory_bundle(path, device='cpu'):
    """Explicit architecture loader: do not load this time network as a legacy dictionary."""
    bundle=torch.load(path,map_location='cpu',weights_only=True,mmap=True)
    if bundle.get('format')!='pi05-trajectory-transcoders-v1' or not bundle.get('complete'):
        raise ValueError('Not a complete trajectory-trained bundle')
    models={}
    for name,cfg in bundle['configs'].items():
        if bundle['architectures'][name]!='diffract_adapted_time_v1': raise ValueError('Unknown architecture')
        model=PaperAdaptedTranscoder(TimeConditionedTranscoderConfig(**cfg))
        model.load_state_dict(bundle['state_dicts'][name],strict=True)
        models[name]=model.to(device).eval()
    return models,bundle


def validation_with_occupancy(model,records,normalizer):
    counts=torch.zeros(model.config.latent_dim,device=next(model.parameters()).device,dtype=torch.int64)
    def count(module,inputs,result):
        counts.add_((result[1]>0).reshape(-1,len(counts)).sum(0))
    handle=model.register_forward_hook(count)
    try: result=reconstruction_metrics(model,records,normalizer)
    finally: handle.remove()
    norms=model.decoder.weight.detach().norm(dim=0)
    result.update(dictionary_features=len(counts),observed_never_fired_fraction=float((counts==0).float().mean()),
        decoder_norm_min=float(norms.min()),decoder_norm_max=float(norms.max()),
        occupancy_scope='Fixed validation/test sample only; not a lifetime dead-feature estimate')
    return result


class TrainingDashboard:
    """Updates display IDs, preserving logs/tracebacks; persists JSONL and a labelled PNG."""
    def __init__(self, root, state_dir, config, enabled=True, refresh_seconds=10):
        self.root,self.state_dir,self.config=Path(root),Path(state_dir),config
        self.root.mkdir(parents=True,exist_ok=True); self.state_dir.mkdir(parents=True,exist_ok=True)
        self.path=self.state_dir/'metrics.jsonl'
        self.rows=[json.loads(s) for s in self.path.read_text().splitlines()] if self.path.exists() else []
        self.enabled=enabled; self.refresh_seconds=refresh_seconds; self.last=0.; self.last_image=0.
        self.banner=None; self.figure=None; self.status={}; self.started=time.monotonic();self.last_status=0.

    def emit(self, phase, *, force=False, **values):
        previous=self.status.get('phase');self.status.update(values); self.status['phase']=phase
        event=dict(event='status',phase=phase,wall_time=time.time(),**values)
        if force or previous!=phase or time.monotonic()-self.last_status>=5:
            audit_json(self.root/'live_status.json',event);self.last_status=time.monotonic()
        if force: print(phase,values,flush=True)
        self.render(force)

    def record(self, row):
        row=dict(row,wall_time=time.time())
        with self.path.open('a') as out: out.write(json.dumps(row,allow_nan=False)+'\n')
        self.rows.append(row); self.render(False)

    def render(self, force=False):
        now=time.monotonic()
        if not self.enabled or (not force and now-self.last<self.refresh_seconds): return
        self.last=now
        from IPython.display import display,HTML
        import html
        import matplotlib.pyplot as plt
        title='Trajectory transcoder training — fidelity metrics; attribution NOT tested'
        fields=' | '.join(f'<b>{html.escape(str(k))}</b>: {html.escape(str(v))}' for k,v in self.status.items())
        box=HTML(f'<h3>{title}</h3><p>{fields}</p><p>Model initialization: fresh. '
            'Training objective uses minibatch variance; plotted comparable nMSE uses a frozen training-reference variance. '
            'Validation is episode-held-out. Actions are predicted, never executed.</p>')
        if self.banner is None: self.banner=display(box,display_id=True)
        else: self.banner.update(box)
        fig,axes=plt.subplots(3,3,figsize=(18,13),constrained_layout=True)
        ax=axes.ravel(); layer=int(self.status.get('layer',self.config.layers[0]))
        train=[r for r in self.rows if r.get('event')=='train' and r['layer']==layer]
        val=[r for r in self.rows if r.get('event')=='validation' and r['layer']==layer]
        steps=[r['step'] for r in train]
        for key,label in [('reference_nmse','Train minibatch / fixed reference variance')]:
            ax[0].plot(steps,[r[key] for r in train],'.-',markersize=3,label=label)
        ax[0].plot([r['step'] for r in val],[r['nmse'] for r in val],'o-',label='Fixed validation frames')
        ax[0].set(title=f'Layer {layer}: comparable reconstruction error ↓',xlabel='Optimizer updates for this layer',ylabel='nMSE (fixed denominator)')
        if train or val: ax[0].legend(fontsize=7)
        ax[1].plot(steps,[r['l0'] for r in train],'.-',markersize=3,label='Training minibatch')
        ax[1].plot([r['step'] for r in val],[r['mean_l0'] for r in val],'o-',label='Validation')
        ax[1].set(title=f'Layer {layer}: active features/token',xlabel='Optimizer updates',ylabel='Count of ReLU activations > 0')
        if train or val: ax[1].legend(fontsize=7)
        for key,label in [('objective_nmse','MSE / minibatch variance'),('weighted_l1','λ × mean sum |features|')]:
            ax[2].plot(steps,[r[key] for r in train],'.-',markersize=3,label=label)
        ax[2].set(title=f'Layer {layer}: objective components',xlabel='Optimizer updates',ylabel='Loss contribution')
        if train: ax[2].legend(fontsize=7)
        latest={r['layer']:r for r in self.rows if r.get('event')=='validation'}
        layers=list(self.config.layers)
        ax[3].barh([str(l) for l in layers],[latest.get(l,{}).get('nmse',np.nan) for l in layers])
        ax[3].set(title='Latest validation by layer (blank = not measured)',xlabel='nMSE ↓',ylabel='Action-expert MLP layer')
        grid=np.full((len(layers),self.config.inference_steps),np.nan)
        for i,l in enumerate(layers):
            times=latest.get(l,{}).get('per_time',{})
            for j,t in enumerate(sorted(times,key=float,reverse=True)[:self.config.inference_steps]): grid[i,j]=times[t]['nmse']
        im=ax[4].imshow(np.ma.masked_invalid(grid),aspect='auto',interpolation='nearest')
        ax[4].set(title='Validation error by denoising call',xlabel='Call index (noise → denoised)',ylabel='Layer')
        ax[4].set_yticks(range(len(layers)),layers)
        ax[4].set_xticks(range(self.config.inference_steps),range(1,self.config.inference_steps+1))
        fig.colorbar(im,ax=ax[4],label='nMSE ↓')
        action={r['layer']:r for r in self.rows if r.get('event')=='validation_action'}
        ax[5].barh([str(l) for l in layers],[100*action.get(l,{}).get('relative_l2',np.nan) for l in layers])
        ax[5].set(title=f'Validation action error (n={self.config.action_validation_frames} frames/layer)',xlabel='% relative L2, first 10 normalized actions (not movement)',ylabel='Layer')
        ax[5].set_yticks(range(len(layers)),layers);ax[5].set_ylim(-.5,len(layers)-.5)
        if not action: ax[5].text(.5,.5,'Not yet measured',ha='center',transform=ax[5].transAxes)
        progress={r['layer']:r['step'] for r in self.rows if r.get('event') in ('train','validation')}
        ax[6].barh([str(l) for l in layers],[progress.get(l,0) for l in layers])
        ax[6].set(title='Completed optimizer updates',xlabel='Updates (each layer has its own count)',ylabel='Layer')
        if self.status.get('updates_per_layer'): ax[6].set_xlim(0,self.status['updates_per_layer'])
        ax[7].plot(steps,[r['records_per_second'] for r in train],'.-',markersize=3)
        ax[7].set(title=f'Layer {layer}: optimizer throughput',xlabel='Optimizer updates',ylabel='Sampled activation records / second')
        ax[8].plot(steps,[r['lr'] for r in train],'.-',markersize=3,label='Learning rate')
        ax[8].set(title=f'Layer {layer}: schedule and gradient norm',xlabel='Optimizer updates',ylabel='Learning rate')
        ax[8].ticklabel_format(axis='y',style='sci',scilimits=(0,0))
        other=ax[8].twinx();other.plot(steps,[r['grad_norm'] for r in train],'.-',markersize=3,color='orange',alpha=.6)
        other.set_ylabel('Gradient L2 norm before clipping',color='orange')
        for a in ax: a.grid(alpha=.15)
        if self.figure is None: self.figure=display(fig,display_id=True)
        else: self.figure.update(fig)
        if force or now-self.last_image>=60:
            temp=self.root/'live_training.tmp.png';fig.savefig(temp,dpi=110);temp.replace(self.root/'live_training.png')
            shutil.copy2(self.path,self.root/'metrics.jsonl') if self.path.exists() else None
            self.last_image=now
        plt.close(fig)


def publish_training_checkpoint(state_dir, root, protocol_hash, dashboard=None):
    """Immutable shard snapshot; latest pointer is committed last, retain two complete snapshots."""
    root,state_dir=Path(root),Path(state_dir)
    name='snapshot_'+uuid.uuid4().hex[:16]; folder=root/'checkpoints'/name;folder.mkdir(parents=True)
    if dashboard: dashboard.emit('saving checkpoint',force=True,checkpoint=name)
    files={}
    for source in sorted(state_dir.iterdir()):
        if source.is_file() and source.suffix in ('.pt','.json','.jsonl'):
            target=folder/source.name;shutil.copy2(source,target);files[source.name]=file_sha256(target)
            if dashboard: dashboard.emit('saving checkpoint',saved_shards=len(files))
    audit_json(folder/'receipt.json',dict(protocol_hash=protocol_hash,files=files))
    pointer=root/'latest.json';old=json.loads(pointer.read_text()) if pointer.exists() else {}
    keep=[name]+old.get('snapshots',[])[:1]
    audit_json(pointer,dict(protocol_hash=protocol_hash,snapshots=keep))
    for previous in old.get('snapshots',[])[1:]:
        if previous.startswith('snapshot_') and Path(previous).name==previous:
            shutil.rmtree(root/'checkpoints'/previous)
    if dashboard: dashboard.emit('checkpoint saved',force=True,checkpoint=name,persistent_path=str(folder))
    return name


def restore_training_checkpoint(root, state_dir, protocol_hash):
    root,state_dir=Path(root),Path(state_dir);state_dir.mkdir(parents=True,exist_ok=True)
    pointer=root/'latest.json'
    if not pointer.exists():
        if any(state_dir.iterdir()): raise ValueError('Local state exists without a committed checkpoint; use a clean local cache')
        return
    latest=json.loads(pointer.read_text())
    if latest['protocol_hash']!=protocol_hash: raise ValueError('Checkpoint protocol changed')
    folder=root/'checkpoints'/latest['snapshots'][0];receipt=json.loads((folder/'receipt.json').read_text())
    if receipt['protocol_hash']!=protocol_hash: raise ValueError('Snapshot protocol changed')
    for name,digest in receipt['files'].items():
        if Path(name).name!=name or file_sha256(folder/name)!=digest: raise ValueError('Corrupt checkpoint shard')
    for path in state_dir.iterdir():
        if path.is_file(): path.unlink()
    for name in receipt['files']: shutil.copy2(folder/name,state_dir/name)


def optimize_trajectory_layer(model, optimizer, records, normalizer, config, layer, cycle, cursor,
                              updates_per_cycle, dashboard, deadline, device):
    """Deterministic per-update sampling makes interrupted and uninterrupted training identical."""
    total=config.cycles*updates_per_cycle;last_time=time.monotonic();last_cursor=cursor
    while cursor<updates_per_cycle and time.monotonic()<deadline:
        step=cycle*updates_per_cycle+cursor
        g=torch.Generator().manual_seed(config.seed+1000003*cycle+cursor)
        indices=torch.randint(len(records['t']),(config.batch_size,),generator=g)
        x,y,t=(records[k][indices].to(device) for k in ('x','y','t'))
        lr=config.lr*.5*(1+math.cos(math.pi*step/total));optimizer.param_groups[0]['lr']=lr
        optimizer.zero_grad(set_to_none=True)
        pred,z=model(x,t)
        raw_mse=(pred.float()-y.float()).square().sum(-1).mean()
        nmse=raw_mse/(y.float().var(0,unbiased=False).sum()+1e-6)
        weighted_l1=config.lambda_l1*z.float().abs().sum(-1).mean();loss=nmse+weighted_l1
        if not torch.isfinite(loss): raise ValueError('Nonfinite loss; checkpoint preserved, native traceback follows')
        loss.backward();grad=torch.nn.utils.clip_grad_norm_(model.parameters(),config.grad_clip,error_if_nonfinite=True)
        optimizer.step();model.renormalize_decoder_columns();cursor+=1
        if cursor%config.log_every==0 or cursor==updates_per_cycle:
            elapsed=max(time.monotonic()-last_time,1e-6)
            dashboard.record(dict(event='train',layer=layer,cycle=cycle+1,step=cycle*updates_per_cycle+cursor,
                reference_nmse=float(raw_mse.detach())/normalizer,objective_nmse=float(nmse.detach()),
                weighted_l1=float(weighted_l1.detach()),loss=float(loss.detach()),l0=float((z>0).sum(-1).float().mean()),
                lr=lr,grad_norm=float(grad),records_per_second=(cursor-last_cursor)*config.batch_size/elapsed))
            last_time=time.monotonic();last_cursor=cursor
        dashboard.emit('training',layer=layer,cycle=f'{cycle+1}/{config.cycles}',
                       optimizer_step=cycle*updates_per_cycle+cursor,
                       gpu_allocated_GiB=round(torch.cuda.memory_allocated()/2**30,2) if str(device).startswith('cuda') else 0)
    return cursor


In [ ]:
# @title Definitions 7 — Dataset Trajectories And Training Loop
"""Full-dataset observation collection, frozen-teacher training, and fidelity evaluation."""

from dataclasses import asdict
import gc
import hashlib
import inspect
import json
import math
from pathlib import Path
import time
from types import SimpleNamespace

import numpy as np
import torch



class DatasetTrajectorySource:
    """Uses whole recorded episodes as observations; policy actions are predicted, never executed."""
    def __init__(self,dataset,policy_path,config,device='cuda'):
        self.dataset,self.config,self.device=dataset,config,device
        args=SimpleNamespace(suite='libero_spatial',task_id=0,prompt='',policy_path=str(policy_path),
                             device=device,policy_dtype='bfloat16')
        self.base=build_harness(args);self.policy=self.base.policy;self.base.vec_env.close()
        self.teacher_load=assert_weights_loaded(self.policy,source=str(policy_path))
        if self.policy.training or any(p.requires_grad for p in self.policy.parameters()):
            raise ValueError('Teacher must be in evaluation mode with all weights frozen')
        targets=list_pi05_action_expert_mlp_targets(self.policy)
        self.targets={t.layer_index:t for t in targets if t.layer_index in config.layers}
        if set(self.targets)!=set(config.layers): raise ValueError('Requested policy layers missing')
        self.widths={l:t.d_model for l,t in self.targets.items()}
        self.names={l:t.name for l,t in self.targets.items()}
        self.chunk_size=int(self.policy.model.config.chunk_size)
        self.handles=[];self.capture=None;self.offsets={};self.replacements={};self.active=False
        self.original=self.policy.model.denoise_step;signature=inspect.signature(self.original)
        def denoise(*args,**kwargs):
            bound=signature.bind(*args,**kwargs)
            self.t=float(bound.arguments['timestep'].float().reshape(-1)[0]);self.active=True
            try: return self.original(*bound.args,**bound.kwargs)
            finally: self.active=False;self.calls+=1
        self.policy.model.denoise_step=denoise
        for layer,target in self.targets.items(): self.handles.append(target.module.register_forward_hook(self.hook(layer)))

    def hook(self,layer):
        def observe(module,inputs,output):
            if not self.active: raise RuntimeError('MLP called outside denoising')
            if self.capture is not None:
                x=inputs[0].detach().reshape(-1,inputs[0].shape[-1]);y=output.detach().reshape_as(x)
                if not torch.isfinite(x).all() or not torch.isfinite(y).all(): raise ValueError('Nonfinite teacher activation')
                offset=self.offsets[layer];end=offset+len(x)
                if end>len(self.capture[layer]['t']): raise ValueError('Activation buffer exceeded expected call count')
                self.capture[layer]['x'][offset:end].copy_(x)
                self.capture[layer]['y'][offset:end].copy_(y)
                self.capture[layer]['t'][offset:end]=self.t;self.offsets[layer]=end
            if layer in self.replacements:
                t=torch.full((inputs[0].shape[0],),self.t,device=inputs[0].device)
                y,_=self.replacements[layer](inputs[0],t)
                if not torch.isfinite(y).all(): raise ValueError('Nonfinite replacement prediction')
                return y.to(output.dtype)
            return output
        return observe

    def batch(self,index):
        from torch.utils.data import default_collate
        row=self.dataset[int(index)]
        if int(row['index'])!=int(index): raise ValueError('Dataset row indices do not match frozen frame plan')
        obs={k:v for k,v in row.items() if k.startswith('observation.')}
        task=row.get('task')
        if not isinstance(task,str) or not task.strip(): raise ValueError('Dataset frame has no task instruction')
        for key in self.dataset.meta.camera_keys:
            if key not in obs: raise ValueError(f'Dataset camera absent: {key}')
            if obs[key].dtype==torch.uint8: obs[key]=obs[key].float()/255.
        obs['task']=task
        return self.base.preprocessor(default_collate([obs]))

    def predict(self,index,seed,replacements=None):
        batch=self.batch(index);self.policy.reset();self.calls=0;self.replacements=replacements or {}
        g=torch.Generator(device=self.device).manual_seed(int(seed))
        noise=torch.randn((1,self.chunk_size,self.policy.model.config.max_action_dim),generator=g,device=self.device,dtype=torch.float32)
        try:
            with torch.inference_mode(): actions=self.policy.predict_action_chunk(batch,noise=noise,num_steps=self.config.inference_steps)
            if self.calls!=self.config.inference_steps: raise ValueError('Wrong denoising call count')
            return actions[0].float().cpu().numpy()
        finally: self.replacements={}

    def collect(self,indices,seed,dashboard,deadline):
        n=len(indices)*self.chunk_size*self.config.inference_steps
        self.capture={l:dict(x=torch.empty(n,d,dtype=torch.float32,device='cpu'),
            y=torch.empty(n,d,dtype=torch.float32,device='cpu'),t=torch.empty(n,dtype=torch.float32,device='cpu')) for l,d in self.widths.items()}
        self.offsets={l:0 for l in self.targets}
        started=time.monotonic()
        try:
            for i,index in enumerate(indices):
                if time.monotonic()>=deadline: raise CollectionBudgetExpired()
                self.predict(index,seed+i)
                dashboard.emit('collecting native trajectories',frames_in_buffer=f'{i+1}/{len(indices)}',
                    layer_count=len(self.targets),collection_frames_per_second=round((i+1)/max(time.monotonic()-started,1e-6),3))
            if any(v!=n for v in self.offsets.values()): raise ValueError('Missing layer activation records')
            result=self.capture;self.capture=None
            return result
        finally: self.capture=None

    def action_metrics(self,indices,seed,replacements):
        rows=[]
        for i,index in enumerate(indices):
            baseline=self.predict(index,seed+i);repeat=self.predict(index,seed+i)
            if not np.array_equal(baseline,repeat): raise ValueError('Native repeat drift: fidelity comparison refused')
            modified=self.predict(index,seed+i,replacements)
            metrics=action_error(baseline,modified,10)
            rows.append(dict(frame=int(index),seed=seed+i,**metrics))
        values=[r['prefix']['relative_l2'] for r in rows]
        return dict(relative_l2=float(np.mean(values)) if values and all(v is not None for v in values) else None,
                    cases=len(rows),repeat_floor=0.,rows=rows)

    def close(self):
        self.policy.model.denoise_step=self.original
        for handle in self.handles: handle.remove()


class CollectionBudgetExpired(Exception): pass


def dataset_episodes(dataset):
    table=dataset.meta.episodes
    if hasattr(table,'to_pandas'): rows=table.to_pandas().to_dict('records')
    elif hasattr(table,'to_dict'): rows=table.to_dict('records')
    else: rows=list(table)
    return [{k:int(row[k]) for k in ('episode_index','dataset_from_index','dataset_to_index')} for row in rows]


def fingerprint_dataset(dataset, callback=print):
    root=Path(dataset.root);files={}
    selected=sorted(p for p in root.rglob('*') if p.is_file() and p.suffix in ('.json','.jsonl','.parquet','.mp4'))
    if not selected: raise ValueError('Dataset has no hashable metadata/data/video files')
    total=sum(p.stat().st_size for p in selected);done=0;last=0.
    for path in selected:
        files[str(path.relative_to(root))]=file_sha256(path);done+=path.stat().st_size
        if time.monotonic()-last>5:
            callback(f'Dataset identity: hashed {done/2**30:.2f}/{total/2**30:.2f} GiB');last=time.monotonic()
    return dict(repo_id=dataset.repo_id,files=files,bytes=total,
                sha256=hashlib.sha256(json.dumps(files,sort_keys=True).encode()).hexdigest())


def layer_progress(state_dir,layer):
    path=Path(state_dir)/f'layer_{layer}.pt'
    if not path.exists(): return None
    return torch.load(path,map_location='cpu',mmap=True,weights_only=True)


def prune_uncommitted_metrics(dashboard,state_dir):
    steps={l:(layer_progress(state_dir,l) or {}).get('step',0) for l in dashboard.config.layers}
    unique={}
    for r in dashboard.rows:
        if r.get('step',0)<=steps.get(r.get('layer'),0): unique[(r.get('event'),r.get('layer'),r.get('step'))]=r
    dashboard.rows=list(unique.values())
    dashboard.path.write_text(''.join(json.dumps(r,allow_nan=False)+'\n' for r in dashboard.rows))


def run_trajectory_training(config,source,episodes,root,local,provenance,*,minutes=120,
                            checkpoint_minutes=30,charts=True,device='cuda'):
    """Train one dictionary at a time on shared fresh buffers; snapshots cover all completed updates."""
    config.validate()
    if minutes<=0 or checkpoint_minutes<=0: raise ValueError('Budgets must be positive')
    started=time.monotonic();deadline=started+minutes*60
    root,local=Path(root),Path(local);root.mkdir(parents=True,exist_ok=True);local.mkdir(parents=True,exist_ok=True)
    plan=training_plan(episodes,config)
    protocol=json.loads(json.dumps(dict(config=asdict(config),provenance=provenance,plan=plan,
        architecture='diffract_adapted_time_v1',collection='native inference trajectories on dataset observations')))
    protocol_hash=hashlib.sha256(json.dumps(protocol,sort_keys=True).encode()).hexdigest()
    manifest=root/'manifest.json'
    if manifest.exists() and json.loads(manifest.read_text())!=protocol:
        raise ValueError('Run protocol/data/teacher changed. Use a new RUN_NAME; existing checkpoints are preserved.')
    audit_json(manifest,protocol)
    finished=root/'training_summary.json'
    if finished.exists() and (root/'trained_transcoders.pt').exists():
        summary=json.loads(finished.read_text())
        if summary.get('status')=='complete' and summary.get('export_sha256')==file_sha256(root/'trained_transcoders.pt'):
            print('This training run and final test are already complete.',flush=True)
            return summary
    if (root/'failure.json').exists():
        history=root/'previous_failures';history.mkdir(exist_ok=True)
        (root/'failure.json').replace(history/f'{time.time_ns()}.json')
    state_dir=local/protocol_hash[:16]/'state';cache=local/protocol_hash[:16]/'cache';cache.mkdir(parents=True,exist_ok=True)
    restore_training_checkpoint(root,state_dir,protocol_hash)
    dashboard=TrainingDashboard(root,state_dir,config,enabled=charts)
    prune_uncommitted_metrics(dashboard,state_dir)
    per_cycle=math.ceil(config.frames_per_cycle*source.chunk_size*config.inference_steps/config.batch_size)
    total_steps=config.cycles*per_cycle
    dashboard.emit('preflight passed',force=True,layers=list(config.layers),updates_per_layer=total_steps,
        samples_per_layer=total_steps*config.batch_size,planned_unique_train_frames=len(set(plan['train_frames'])),
        activation_buffer_GiB=round(config.frames_per_cycle*source.chunk_size*config.inference_steps*
                                   sum(source.widths.values())*8/2**30,2),scope='Fidelity only; attribution untested')
    last_checkpoint=time.monotonic();saved=False
    def checkpoint():
        nonlocal last_checkpoint,saved
        prune_uncommitted_metrics(dashboard,state_dir)
        publish_training_checkpoint(state_dir,root,protocol_hash,dashboard)
        last_checkpoint=time.monotonic();saved=True
    def cached_records(tag,indices,seed):
        path=cache/(tag+'.pt');receipt=cache/(tag+'.json')
        if path.exists() and receipt.exists() and file_sha256(path)==json.loads(receipt.read_text())['sha256']:
            return torch.load(path,map_location='cpu',mmap=True,weights_only=True)
        dashboard.emit('collecting '+tag,force=True,observations=len(indices))
        records=source.collect(indices,seed,dashboard,deadline)
        atomic_torch_save(records,path);audit_json(receipt,dict(sha256=file_sha256(path)))
        return records
    try:
        normalizer_path=state_dir/'normalizers.json'
        if normalizer_path.exists():
            normalizers={int(k):float(v) for k,v in json.loads(normalizer_path.read_text()).items()}
            if set(normalizers)!=set(config.layers): raise ValueError('Saved reference layers changed')
        else:
            reference=cached_records('train_reference',plan['reference_frames'],config.seed+700)
            normalizers={l:float(reference[l]['y'].double().var(0,unbiased=False).sum()) for l in config.layers}
            del reference
        if any(not np.isfinite(v) or v<=1e-12 for v in normalizers.values()): raise ValueError('Degenerate reference variance')
        if not normalizer_path.exists():
            audit_json(normalizer_path,{str(k):v for k,v in normalizers.items()})
        validation=cached_records('validation',plan['validation_frames'],config.seed+800)
        for cycle in range(config.cycles):
            validate_cycle=(cycle+1)%config.validate_every==0 or cycle==0 or cycle+1==config.cycles
            def needs_work(layer):
                trained=(layer_progress(state_dir,layer) or {}).get('step',0)
                marker=state_dir/f'validated_{layer}.json'
                validated=json.loads(marker.read_text())['step'] if marker.exists() else -1
                return trained<(cycle+1)*per_cycle or (validate_cycle and validated<(cycle+1)*per_cycle)
            needed=[l for l in config.layers if needs_work(l)]
            if not needed: continue
            if time.monotonic()>=deadline: checkpoint();return dict(status='paused_budget',root=str(root))
            indices=plan['train_frames'][cycle*config.frames_per_cycle:(cycle+1)*config.frames_per_cycle]
            cycle_started=time.monotonic()
            dashboard.emit('starting cycle',force=True,cycle=f'{cycle+1}/{config.cycles}',
                           unique_train_frames_seen=len(set(plan['train_frames'][:cycle*config.frames_per_cycle])))
            records=cached_records(f'cycle_{cycle}',indices,config.seed+10000+cycle*config.frames_per_cycle)
            for layer in needed:
                if time.monotonic()>=deadline: checkpoint();return dict(status='paused_budget',root=str(root))
                model=new_trajectory_model(source.widths[layer],config,layer).to(device)
                optimizer=torch.optim.AdamW(model.parameters(),lr=config.lr,weight_decay=0.)
                previous=layer_progress(state_dir,layer);cursor=0
                if previous:
                    model.load_state_dict(previous['model'],strict=True);optimizer.load_state_dict(previous['optimizer'])
                    cursor=previous['step']-cycle*per_cycle
                    if not 0<=cursor<=per_cycle: raise ValueError('Layer checkpoint has inconsistent cycle position')
                del previous
                dashboard.emit('training',force=True,layer=layer,cycle=f'{cycle+1}/{config.cycles}',
                               dictionary_features=source.widths[layer]*config.expansion)
                # Only a successfully returned update block is committed. Interrupts discard its uncommitted updates.
                try:
                    if cycle==0 and cursor==0:
                        metrics=validation_with_occupancy(model,validation[layer],normalizers[layer])
                        dashboard.record(dict(event='validation',layer=layer,step=0,cycle=0,**metrics))
                    cursor=optimize_trajectory_layer(model,optimizer,records[layer],normalizers[layer],config,
                        layer,cycle,cursor,per_cycle,dashboard,deadline,device)
                    step=cycle*per_cycle+cursor
                    atomic_torch_save(dict(model=model.state_dict(),optimizer=optimizer.state_dict(),step=step,
                        config=asdict(model.config),protocol_hash=protocol_hash),state_dir/f'layer_{layer}.pt')
                    if cursor==per_cycle and validate_cycle:
                        metrics=validation_with_occupancy(model,validation[layer],normalizers[layer])
                        dashboard.record(dict(event='validation',layer=layer,step=step,cycle=cycle+1,**metrics))
                        dashboard.emit('validation',validation_nmse=round(metrics['nmse'],5),
                            validation_active_features=round(metrics['mean_l0'],1),
                            never_fired_in_validation_percent=round(100*metrics['observed_never_fired_fraction'],2))
                        best_path=state_dir/f'best_{layer}.pt'
                        best=torch.load(best_path,map_location='cpu',mmap=True,weights_only=True) if best_path.exists() else None
                        if best is None or metrics['nmse']<best['validation_nmse']:
                            atomic_torch_save(dict(model=model.state_dict(),config=asdict(model.config),step=step,
                                validation_nmse=metrics['nmse'],validation_l0=metrics['mean_l0']),best_path)
                        del best
                        if config.action_validation_frames:
                            if time.monotonic()>=deadline: raise CollectionBudgetExpired()
                            action=source.action_metrics(plan['validation_frames'][:config.action_validation_frames],
                                config.seed+800,{layer:model})
                            if action['relative_l2'] is not None:
                                dashboard.record(dict(event='validation_action',layer=layer,step=step,cycle=cycle+1,**action))
                        audit_json(state_dir/f'validated_{layer}.json',dict(step=step))
                finally:
                    del model,optimizer;gc.collect()
                    if str(device).startswith('cuda'): torch.cuda.empty_cache()
                if time.monotonic()-last_checkpoint>=checkpoint_minutes*60:
                    checkpoint()
            del records
            # A completed cycle can be regenerated from the frozen frame/noise plan if needed.
            (cache/f'cycle_{cycle}.pt').unlink(missing_ok=True);(cache/f'cycle_{cycle}.json').unlink(missing_ok=True)
            cycle_seconds=time.monotonic()-cycle_started
            dashboard.emit('cycle complete',force=True,cycle=f'{cycle+1}/{config.cycles}',
                observed_cycle_seconds=round(cycle_seconds,1),
                rough_remaining_hours=round(cycle_seconds*(config.cycles-cycle-1)/3600,2),
                unique_train_frames_seen=len(set(plan['train_frames'][:(cycle+1)*config.frames_per_cycle])))
        checkpoint()
        bundle=dict(format='pi05-trajectory-transcoders-v1',complete=True,configs={},state_dicts={},architectures={},
                    manifest=protocol,protocol_hash=protocol_hash,updates_per_layer=total_steps,
                    selection='best validation nMSE per layer',selected_steps={})
        for layer in config.layers:
            saved_layer=layer_progress(state_dir,layer)
            if saved_layer['step']!=total_steps: raise ValueError('Refusing to export an incomplete layer set')
            saved_layer=torch.load(state_dir/f'best_{layer}.pt',map_location='cpu',mmap=True,weights_only=True)
            name=source.names[layer];bundle['configs'][name]=saved_layer['config']
            bundle['state_dicts'][name]=saved_layer['model'];bundle['architectures'][name]='diffract_adapted_time_v1'
            bundle['selected_steps'][name]=saved_layer['step']
        dashboard.emit('exporting trained dictionaries',force=True)
        atomic_torch_save(bundle,root/'trained_transcoders.pt');del bundle
        test=cached_records('test',plan['test_frames'],config.seed+900)
        results={}
        for layer in config.layers:
            path=root/'test'/f'layer_{layer}.json'
            if path.exists(): results[str(layer)]=json.loads(path.read_text());continue
            if time.monotonic()>=deadline:
                audit_json(root/'training_summary.json',dict(status='training_complete_test_pending',updates_per_layer=total_steps))
                return dict(status='training_complete_test_pending',root=str(root))
            state=torch.load(state_dir/f'best_{layer}.pt',map_location='cpu',mmap=True,weights_only=True)
            model=new_trajectory_model(source.widths[layer],config,layer).to(device)
            model.load_state_dict(state['model']);model.eval();del state
            metrics=validation_with_occupancy(model,test[layer],normalizers[layer])
            action=source.action_metrics(plan['test_frames'],config.seed+900,{layer:model})
            results[str(layer)]=dict(reconstruction=metrics,action=action,frames=len(plan['test_frames']))
            path.parent.mkdir(exist_ok=True);audit_json(path,results[str(layer)]);del model
            dashboard.emit('final test evaluation',force=True,layer=layer,test_layers_complete=len(results))
        summary=dict(status='complete',protocol_hash=protocol_hash,export_sha256=file_sha256(root/'trained_transcoders.pt'),
            layers=list(config.layers),updates_per_layer=total_steps,
            sampled_records_per_layer=total_steps*config.batch_size,unique_train_frames=len(set(plan['train_frames'])),
            episode_split_counts={k:len(v) for k,v in plan['episode_ids'].items()},test=results,
            limitations=['Episode-held-out observations, not unseen tasks or policy-pretraining holdout.',
                'Export selects the best measured validation nMSE separately per layer; layers may come from different updates.',
                'Per-layer normalized-action fidelity only; simultaneous all-layer replacement not measured.',
                'No feature attribution, steering, executed task success or safety claim.',
                'Adapted VLA recipe and finite data budget, not a full DifFRACT reproduction.'])
        audit_json(root/'training_summary.json',summary)
        dashboard.emit('complete',force=True,export=str(root/'trained_transcoders.pt'))
        return summary
    except CollectionBudgetExpired:
        checkpoint();dashboard.emit('paused: session budget reached',force=True)
        return dict(status='paused_budget',root=str(root))
    except BaseException as exc:
        audit_json(root/'failure.json',dict(type=type(exc).__name__,message=str(exc)))
        try: checkpoint();dashboard.emit('interrupted; last committed updates saved',force=True,error=str(exc))
        except Exception as save_error: print('Checkpoint save also failed:',save_error,flush=True)
        raise
    finally:
        audit_json(root/'last_session.json',dict(elapsed_seconds=time.monotonic()-started,requested_minutes=minutes))
        dashboard.render(True)

TRAJECTORY_TRAINING_VERSION = 'f65188e36f509395c922fa8c4c2a06f750440deef651870120e87dc440a9f156'


In [ ]:
# @title Train / Resume — Live Dashboard Below, Native Tracebacks Preserved
from importlib.metadata import version
from IPython.display import display,Markdown
import gc
if 'TRAJECTORY_TRAINING_VERSION' not in globals(): raise RuntimeError('Run all seven Definitions cells first')
config=TrajectoryTrainConfig(layers=tuple(int(x.strip()) for x in LAYER_IDS.split(',') if x.strip()),
    cycles=CYCLES,frames_per_cycle=FRAMES_PER_CYCLE,batch_size=TRAIN_BATCH_SIZE,
    lambda_l1=LAMBDA_L1,validate_every=VALIDATE_EVERY_CYCLES)
config.validate()
print('Notebook definitions:',TRAJECTORY_TRAINING_VERSION[:12])
print('Hashing teacher and dataset bytes to freeze training provenance...',flush=True)
provenance=dict(source_digest=TRAJECTORY_TRAINING_VERSION,
    teacher=snapshot_evidence(snapshots[POLICY_REPO]),dataset=fingerprint_dataset(dataset),
    packages={p:version(p) for p in ('torch','transformers','lerobot','hf-libero','numpy','matplotlib','av')},
    snapshots=snapshots,gpu=torch.cuda.get_device_name(0))
source=None
try:
    source=DatasetTrajectorySource(dataset,snapshots[POLICY_REPO],config)
    audit_json(RUN_DIR/'teacher_load.json',source.teacher_load)
    result=run_trajectory_training(config,source,dataset_episodes(dataset),RUN_DIR,
        '/content/pi05-trajectory-training',provenance,minutes=SESSION_MINUTES,
        checkpoint_minutes=CHECKPOINT_EVERY_MINUTES)
    print('Session result:',result['status'])
    print('Metrics:',RUN_DIR/'metrics.jsonl')
    print('Saved dashboard:',RUN_DIR/'live_training.png')
finally:
    if source is not None: source.close()
    del source;gc.collect();torch.cuda.empty_cache()


In [ ]:
# @title Inspect Checkpoint Status And Download Small Training Evidence
import json,zipfile
from IPython.display import display,FileLink,Image
for name in ('live_status.json','latest.json','training_summary.json','failure.json'):
    path=RUN_DIR/name
    if path.exists():
        value=json.loads(path.read_text())
        print(name,':',value if name not in ('training_summary.json',) else {k:v for k,v in value.items() if k!='test'})
if (RUN_DIR/'live_training.png').exists(): display(Image(filename=str(RUN_DIR/'live_training.png'),width=1200))
packet=Path('/content')/(RUN_NAME+'-evidence.zip')
with zipfile.ZipFile(packet,'w',zipfile.ZIP_DEFLATED) as archive:
    for path in RUN_DIR.rglob('*'):
        if 'checkpoints' not in path.relative_to(RUN_DIR).parts and path.is_file() and path.suffix in ('.json','.jsonl','.png'):
            archive.write(path,path.relative_to(RUN_DIR))
display(FileLink(str(packet)))
print('Weights and optimizer shards remain on Drive; only small evidence is zipped.')
print('Complete exports: trained_transcoders.pt — use load_trajectory_bundle(path), which handles the new time-network architecture.')
